In [2]:
"""
map the CAMELS-CH download before building anything.

Usage:
    python 00_inspect_camels.py /path/to/camels_ch

Prints:
  1. Folder tree with file counts 
  2. Header row of one representative file per distinct filename pattern
  3. Gauging-station table, with any lake/river type field highlighted
  4. Human-influence attributes (regulation filter)
  5. Water-level completeness per candidate lake gauge

"""

import re
import sys
from collections import defaultdict
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

SEP_GUESSES = [";", ",", "\t"]


def rule(title):
    print(f"\n{'=' * 78}\n{title}\n{'=' * 78}")


def read_csv_smart(path, **kw):
    """CAMELS-CH uses ';'. try all separators"""
    for sep in SEP_GUESSES:
        try:
            df = pd.read_csv(path, sep=sep, **kw)
            if df.shape[1] > 1:
                return df
        except Exception:
            continue
    return pd.read_csv(path, sep=";", engine="python", **kw)


def pattern_of(name):
    """Collapses 'CAMELS_CH_obs_based_2011.csv' -> 'CAMELS_CH_obs_based_<ID>.csv'."""
    return re.sub(r"\d+", "<ID>", name)


# ---------------------------------------------------------------- 1. structure
def show_tree(root):
    rule("1. FOLDER STRUCTURE")
    for folder in sorted(p for p in root.rglob("*") if p.is_dir()):
        files = [f for f in folder.iterdir() if f.is_file()]
        if not files:
            continue
        rel = folder.relative_to(root)
        patterns = defaultdict(list)
        for f in files:
            patterns[pattern_of(f.name)].append(f)
        print(f"\n{rel}/   ({len(files)} files)")
        for pat, group in sorted(patterns.items()):
            print(f"    {pat:<52} x{len(group)}")


# ------------------------------------------------------------------ 2. headers
def show_headers(root):
    rule("2. COLUMN HEADERS (one sample per file pattern)")
    seen = set()
    for f in sorted(root.rglob("*.csv")):
        pat = (f.parent.relative_to(root).as_posix(), pattern_of(f.name))
        if pat in seen:
            continue
        seen.add(pat)
        print(f"\n--- {pat[0]}/{pat[1]}")
        print(f"    (sample: {f.name})")
        try:
            head = read_csv_smart(f, nrows=3)
            print(f"    rows sampled: {head.shape[0]}, cols: {head.shape[1]}")
            for c in head.columns:
                print(f"      - {c}")
        except Exception as e:
            print(f"    !! could not parse: {e}")


# ------------------------------------------------------- 3. gauging stations
def load_dbf(path):
    """Read a .dbf via geopandas if available, else pyshp, else give up"""
    try:
        import geopandas as gpd
        return pd.DataFrame(gpd.read_file(path).drop(columns="geometry", errors="ignore"))
    except Exception:
        pass
    try:
        import shapefile  # pyshp
        r = shapefile.Reader(str(path.with_suffix("")))
        cols = [f[0] for f in r.fields[1:]]
        return pd.DataFrame(r.records(), columns=cols)
    except Exception as e:
        print(f"    !! install geopandas or pyshp to read {path.name}  ({e})")
        return None


def show_stations(root):
    rule("3. GAUGING STATIONS  <- find your lakes here")
    hits = list(root.rglob("*gauging_stations*.dbf")) + list(root.rglob("*gauging_stations*.shp"))
    if not hits:
        print("    no gauging_stations file found")
        return None

    df = load_dbf(hits[0])
    if df is None:
        return None

    print(f"    source: {hits[0].name}   shape: {df.shape}\n")
    print("    columns:", list(df.columns), "\n")

    # Surface any low-cardinality text field - one of these separates lake from river.
    print("    Categorical fields (candidates for the lake/river flag):")
    for c in df.columns:
        vals = df[c].dropru = df[c].dropna().unique() if False else df[c].dropna().unique()
        if 1 < len(vals) <= 12:
            print(f"      {c:<22} -> {list(vals)[:12]}")

    print("\n    First 5 rows:")
    print(df.head().to_string(index=False))
    return df


# ------------------------------------------------------- 4. human influence
def show_human_influence(root):
    rule("4. HUMAN INFLUENCE ATTRIBUTES  <- your regulation filter")
    hits = [p for p in root.rglob("*.csv") if "human" in p.name.lower()]
    if not hits:
        print("    none found - check static_attributes/ filenames in section 2")
        return None
    df = read_csv_smart(hits[0])
    print(f"    source: {hits[0].name}   shape: {df.shape}\n")
    print("    columns:", list(df.columns), "\n")
    for c in df.columns:
        vals = df[c].dropna().unique()
        if 1 < len(vals) <= 12:
            print(f"      {c:<28} -> {list(vals)[:12]}")
    print("\n", df.head().to_string(index=False))
    return df


# ------------------------------------------------- 5. water-level completeness
def show_completeness(root, station_df, limit=40):
    rule("5. WATER-LEVEL COMPLETENESS PER GAUGE")
    ts_dirs = [d for d in root.rglob("*") if d.is_dir() and "time_series" in d.name.lower()]
    if not ts_dirs:
        print("    no time_series folder found")
        return

    files = []
    for d in ts_dirs:
        files.extend(f for f in d.rglob("*.csv") if "obs" in f.name.lower())
    if not files:
        files = [f for d in ts_dirs for f in d.rglob("*.csv")]
    print(f"    scanning {len(files)} daily files (showing gauges with water level)\n")

    rows = []
    for f in files:
        m = re.search(r"(\d{3,})", f.name)
        if not m:
            continue
        gid = m.group(1)
        try:
            df = read_csv_smart(f)
        except Exception:
            continue

        lvl = [c for c in df.columns if "level" in c.lower()]
        if not lvl:
            continue
        col = lvl[0]
        datecol = next((c for c in df.columns if "date" in c.lower()), df.columns[0])
        s = pd.to_numeric(df[col], errors="coerce")
        rows.append({
            "gauge_id": gid,
            "level_col": col,
            "n_rows": len(s),
            "n_valid": int(s.notna().sum()),
            "pct_complete": round(100 * s.notna().mean(), 1),
            "start": str(df[datecol].iloc[0])[:10],
            "end": str(df[datecol].iloc[-1])[:10],
        })

    if not rows:
        print("    !! no water-level column found in any file.")
        print("       Check section 2 for the real column name and adjust the filter.")
        return

    out = pd.DataFrame(rows).sort_values("pct_complete", ascending=False)
    print(f"    {len(out)} gauges carry a water-level series.\n")
    print(out.head(limit).to_string(index=False))

    out.to_csv("water_level_inventory.csv", index=False)
    print("\n    -> written: water_level_inventory.csv")
    print("       Cross-reference gauge_id against sections 3 and 4 to pick your lakes.")


def main():
    if len(sys.argv) < 2:
        sys.exit("usage: python 00_inspect_camels.py /path/to/camels_ch")
    root = Path(sys.argv[1]).expanduser().resolve()
    if not root.is_dir():
        sys.exit(f"not a directory: {root}")

    print(f"CAMELS-CH inspection\nroot: {root}")

    readme = next((p for p in root.rglob("readme*.txt")), None)
    if readme:
        rule("0. README (first 60 lines)")
        for line in readme.read_text(errors="replace").splitlines()[:60]:
            print("   ", line)

    show_tree(root)
    show_headers(root)
    stations = show_stations(root)
    show_human_influence(root)
    show_completeness(root, stations)

    rule("NEXT")
    print("""
    1. From section 3, note the field that flags lake vs river stations.
    2. From section 4, note the regulation / human-influence field.
    3. From water_level_inventory.csv, keep lake gauges with pct_complete >= 90.
    """)


if __name__ == "__main__":
    ROOT = Path(r"C:\Users\prave\Documents\UoMDS\DISSERTATION\camels_ch")

    readme = next((p for p in ROOT.rglob("readme*.txt")), None)
    if readme:
        rule("0. README (first 60 lines)")
        for line in readme.read_text(errors="replace").splitlines()[:60]:
            print("   ", line)
    
    show_tree(ROOT)
    show_headers(ROOT)
    stations = show_stations(ROOT)
    show_human_influence(ROOT)
    show_completeness(ROOT, stations)


0. README (first 60 lines)
    # -----------------------------------------------------------------------------------------
    CAMELS-CH: hydrometeorological time series and landscape attributes for 331 catchments in hydrologic Switzerland
    March 2023
    
    Authors: marvin.hoege@eawag.ch; martina.kauzlaric@giub.unibe.ch; rosi.siber@eawag.ch; ursula.schoenenberger@eawag.ch; pascal.horton@giub.unibe.ch; 
    	jan.schwanbeck@giub.unibe.ch; sibylle.wilhelm@giub.unibe.ch; daniel.viviroli@geo.uzh.ch; anna.senoner@c2sm.ethz.ch; floriancic@ifu.baug.ethz.ch; 
    	manuela.brunner@slf.ch; sandra.pool@eawag.ch; massimiliano.zappa@wsl.ch; fabrizio.fenicia@eawag.ch
    
    Contact: marvin.hoege@eawag.ch
    
    # -----------------------------------------------------------------------------------------
    
    Folders and content:
    
    - timeseries: daily time series of all hydrometeorological variables in subfolders
    	- observation_based: all variables based on observations by BAFU

In [3]:
"""
build the curated natural-lake dataset from CAMELS-CH.

Notebook:   build(r"C:\\Users\\prave\\Documents\\UoMDS\\DISSERTATION\\camels_ch")
Terminal:   python 01_build_dataset.py "C:\\...\\camels_ch"

Outputs (into ./output):
    lakes_static.csv        one row per lake: attributes + regulation tier
    lakes_daily.csv         the daily panel: raw + QC flags + derived features
    data_dictionary.csv     every column, unit, source, derivation
    qc_report.csv           per-lake completeness and flag counts
    natural_lakes.xlsx      the Excel deliverable (4 sheets)

"""

import re
import sys
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

# ---------------------------------------------------------------- parameters

MIN_COMPLETENESS = 90.0    # % non-missing water level required
FLATLINE_RUN     = 5       # identical consecutive values = stuck sensor
MAD_WINDOW       = 30      # days, rolling window for spike detection
MAD_THRESHOLD    = 5.0     # modified z-score above which a point is a spike
GAP_SHORT        = 3       # <= this many days: linear interpolation
GAP_MEDIUM       = 14      # <= this many days: seasonal-naive + interpolation
                           # > GAP_MEDIUM: left as NaN and flagged
SPECIFIC_STORAGE_N1 = 10.0 # mm; upstream reservoir capacity per unit catchment

OUT = Path("output")


# ------------------------------------------------------------------ plumbing
def rule(t):
    print(f"\n{'=' * 78}\n{t}\n{'=' * 78}")


def read_semicolon(path, skip_comment=True):
    """CAMELS-CH static files: ';' separated, one '#' comment line, mixed encoding."""
    for enc in ("utf-8", "latin-1", "cp1252"):
        try:
            with open(path, encoding=enc) as fh:
                first = fh.readline()
            skip = 1 if (skip_comment and first.lstrip().startswith("#")) else 0
            df = pd.read_csv(path, sep=";", skiprows=skip, encoding=enc)
            if df.shape[1] > 1:
                return df
        except (UnicodeDecodeError, pd.errors.ParserError):
            continue
    raise IOError(f"could not parse {path.name}")


def find_col(df, *keywords, required=False, label=""):
    """Locate a column by keyword"""
    def norm(s):
        s = unicodedata.normalize("NFKD", str(s))
        return "".join(c for c in s if not unicodedata.combining(c)).lower()

    for kw in keywords:
        for c in df.columns:
            if norm(kw) in norm(c):
                return c
    if required:
        raise KeyError(f"no column matching {keywords} for {label}. Columns: {list(df.columns)}")
    return None


def load_stations(root):
    hits = list(root.rglob("*gauging_stations*.dbf"))
    if not hits:
        raise FileNotFoundError("CAMELS_CH_gauging_stations.dbf not found")
    path = hits[0]
    try:
        import geopandas as gpd
        gdf = gpd.read_file(path)
        df = pd.DataFrame(gdf.drop(columns="geometry", errors="ignore"))
        # keep coordinates if present
        try:
            pts = gdf.geometry.to_crs(4326)
            df["lon"], df["lat"] = pts.x.values, pts.y.values
        except Exception:
            pass
    except ImportError:
        import shapefile
        r = shapefile.Reader(str(path.with_suffix("")))
        df = pd.DataFrame(r.records(), columns=[f[0] for f in r.fields[1:]])
    df["gauge_id"] = pd.to_numeric(df["gauge_id"], errors="coerce").astype("Int64")
    return df


# ----------------------------------------------------------- 1. lake selection
def select_lakes(root):
    rule("1. SELECTING LAKES")
    st = load_stations(root)
    lakes = st[(st["type"].str.lower() == "lake") & (st["country"] == "CH")].copy()
    print(f"    {len(st)} stations -> {len(lakes)} Swiss lake stations")
    return lakes.reset_index(drop=True)


# ------------------------------------------------------- 2. regulation tiering
def classify_regulation(root, lakes):
    """
    Operational definition of a NATURAL water reservoir.

      N0  no upstream hydropower and no upstream reservoir
      N1  some upstream storage, but specific capacity < SPECIFIC_STORAGE_N1 mm
      R   substantial upstream anthropogenic storage

    NOTE: this measures UPSTREAM catchment storage influence.
    It does not capture outlet regulation of the lake body itself.
    """
    rule("2. REGULATION TIERING")
    hi = read_semicolon(next(root.rglob("*humaninfluence*.csv")))
    topo = read_semicolon(next(root.rglob("*topographic_attributes.csv")))

    hi["gauge_id"] = pd.to_numeric(hi["gauge_id"], errors="coerce").astype("Int64")
    topo["gauge_id"] = pd.to_numeric(topo["gauge_id"], errors="coerce").astype("Int64")

    area_col = find_col(topo, "area", required=True, label="catchment area")
    elev_col = find_col(topo, "elev")
    slope_col = find_col(topo, "slope")
    print(f"    area column:  {area_col}")
    print(f"    elev column:  {elev_col}")

    keep_topo = ["gauge_id", area_col] + [c for c in (elev_col, slope_col) if c]
    df = lakes.merge(hi, on="gauge_id", how="left").merge(
        topo[keep_topo], on="gauge_id", how="left")

    df = df.rename(columns={area_col: "catchment_area_km2"})
    if elev_col:
        df = df.rename(columns={elev_col: "mean_elevation_m"})
    if slope_col:
        df = df.rename(columns={slope_col: "mean_slope"})

    for c in ("hp_count", "num_reservoir", "reservoir_cap"):
        df[c] = pd.to_numeric(df.get(c), errors="coerce").fillna(0.0)

    # reservoir capacity (m3) spread over the catchment, expressed in mm
    area_m2 = pd.to_numeric(df["catchment_area_km2"], errors="coerce") * 1e6
    df["specific_storage_mm"] = (df["reservoir_cap"] / area_m2 * 1000).replace(
        [np.inf, -np.inf], np.nan).round(3)

    def tier(r):
        if r["hp_count"] == 0 and r["num_reservoir"] == 0:
            return "N0"
        if pd.notna(r["specific_storage_mm"]) and r["specific_storage_mm"] < SPECIFIC_STORAGE_N1:
            return "N1"
        return "R"

    df["regulation_tier"] = df.apply(tier, axis=1)
    print("\n" + df["regulation_tier"].value_counts().to_string())
    print("\n", df[["gauge_id", "water_body", "regulation_tier",
                    "hp_count", "num_reservoir", "specific_storage_mm"]]
          .sort_values("regulation_tier").to_string(index=False))
    return df


# ---------------------------------------------------- 3. attach more attributes
def attach_attributes(root, static):
    rule("3. ATTACHING STATIC ATTRIBUTES")
    wanted = {
        "glacier": ["glac_area", "glac_perc", "glacier"],
        "climate": ["p_mean", "t_mean", "frac_snow", "aridity"],
        "landcover": ["ice_perc", "rock_perc", "inwater_perc", "crop_perc"],
    }
    for key, cols in wanted.items():
        hits = [p for p in (root / "static_attributes").glob("*.csv") if key in p.name.lower()]
        if not hits:
            continue
        try:
            df = read_semicolon(hits[0])
        except IOError:
            print(f"    !! skipped {hits[0].name}")
            continue
        df["gauge_id"] = pd.to_numeric(df["gauge_id"], errors="coerce").astype("Int64")
        found = [c for c in df.columns
                 if c != "gauge_id" and any(k in c.lower() for k in cols)]
        if found:
            static = static.merge(df[["gauge_id"] + found], on="gauge_id", how="left")
            print(f"    {hits[0].name}: +{len(found)} cols -> {found}")
    return static


# ------------------------------------------------------------------- 4. QC
def qc_series(s):
    """Return (cleaned series, flag series). Flags: ok / spike / flatline / gap_long."""
    flag = pd.Series("ok", index=s.index, dtype=object)
    flag[s.isna()] = "missing"

    # stuck sensor: runs of identical values
    same = s.eq(s.shift()) & s.notna()
    grp = (~same).cumsum()
    runlen = same.groupby(grp).transform("sum") + 1
    flag[(runlen >= FLATLINE_RUN) & s.notna()] = "flatline"

    # spikes via rolling median absolute deviation (robust to skew, unlike z-scores)
    med = s.rolling(MAD_WINDOW, center=True, min_periods=5).median()
    mad = (s - med).abs().rolling(MAD_WINDOW, center=True, min_periods=5).median()
    mz = 0.6745 * (s - med) / mad.replace(0, np.nan)
    flag[(mz.abs() > MAD_THRESHOLD) & s.notna()] = "spike"

    clean = s.copy()
    clean[flag.isin(["spike", "flatline"])] = np.nan

    # gap taxonomy
    isna = clean.isna()
    gid = (~isna).cumsum()
    gaplen = isna.groupby(gid).transform("sum")

    short = isna & (gaplen <= GAP_SHORT)
    medium = isna & (gaplen > GAP_SHORT) & (gaplen <= GAP_MEDIUM)
    long_ = isna & (gaplen > GAP_MEDIUM)

    filled = clean.interpolate(limit=GAP_SHORT, limit_area="inside")
    if medium.any():
        doy_clim = clean.groupby(clean.index.dayofyear).transform("mean")
        filled[medium] = doy_clim[medium]
        filled = filled.interpolate(limit=GAP_MEDIUM, limit_area="inside")
    filled[long_] = np.nan          # never fabricate the target across long gaps

    flag[short & filled.notna()] = "interp_short"
    flag[medium & filled.notna()] = "interp_seasonal"
    flag[long_] = "gap_long"
    return filled, flag


# -------------------------------------------------------- 5. feature building
def build_features(df):
    """Derived hydrological features. This is what makes the dataset yours."""
    d = df.sort_values("date").copy()
    p, t, swe, lvl = "precipitation_mm", "temperature_mean_c", "swe_mm", "waterlevel_m"

    # Antecedent Precipitation Index: exponentially weighted recent rainfall
    for hl in (30, 60, 90):
        d[f"api_{hl}"] = d[p].ewm(halflife=hl, min_periods=1).mean()

    # Snowmelt proxy: SWE lost on days warm enough to melt
    swe_drop = (d[swe].shift(1) - d[swe]).clip(lower=0)
    d["snowmelt_proxy"] = swe_drop.where(d[t] > 0, 0.0)
    d["snowmelt_cum30"] = d["snowmelt_proxy"].rolling(30, min_periods=1).sum()

    # Thermal forcing
    d["degree_days"] = d[t].clip(lower=0)
    d["cdd_30"] = d["degree_days"].rolling(30, min_periods=1).sum()

    # Autoregressive structure
    for lag in (1, 3, 7, 14, 30):
        d[f"level_lag{lag}"] = d[lvl].shift(lag)
    for w in (7, 30):
        d[f"level_roll{w}_mean"] = d[lvl].shift(1).rolling(w, min_periods=1).mean()
        d[f"level_roll{w}_std"] = d[lvl].shift(1).rolling(w, min_periods=1).std()
    d["level_delta1"] = d[lvl].diff()

    # Seasonality without a January discontinuity
    doy = d["date"].dt.dayofyear
    d["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
    d["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)
    return d


# ------------------------------------------------------------ 6. daily panel
def build_panel(root, static):
    rule("4-5. DAILY PANEL, QC AND FEATURES")
    ts_dir = next(d for d in root.rglob("observation_based") if d.is_dir())
    frames, qc_rows = [], []

    for _, row in static.iterrows():
        gid = int(row["gauge_id"])
        f = ts_dir / f"CAMELS_CH_obs_based_{gid}.csv"
        if not f.exists():
            print(f"    !! missing {f.name}")
            continue

        raw = pd.read_csv(f, sep=";", encoding="latin-1")
        raw.columns = [c.strip() for c in raw.columns]
        ren = {
            find_col(raw, "date", required=True, label=gid): "date",
            find_col(raw, "waterlevel", required=True, label=gid): "waterlevel_m",
            find_col(raw, "precipitation"): "precipitation_mm",
            find_col(raw, "temperature_min"): "temperature_min_c",
            find_col(raw, "temperature_mean"): "temperature_mean_c",
            find_col(raw, "temperature_max"): "temperature_max_c",
            find_col(raw, "swe"): "swe_mm",
            find_col(raw, "rel_sun"): "rel_sun_dur_pct",
            find_col(raw, "discharge_vol"): "discharge_m3s",
        }
        raw = raw.rename(columns={k: v for k, v in ren.items() if k})
        raw["date"] = pd.to_datetime(raw["date"], errors="coerce")
        raw = raw.dropna(subset=["date"]).set_index("date").sort_index()

        for c in ("waterlevel_m", "precipitation_mm", "temperature_mean_c", "swe_mm"):
            if c not in raw:
                raw[c] = np.nan
            raw[c] = pd.to_numeric(raw[c], errors="coerce")

        pct = 100 * raw["waterlevel_m"].notna().mean()
        if pct < MIN_COMPLETENESS:
            print(f"    {gid} {row['water_body'][:24]:<24} dropped ({pct:.1f}% complete)")
            continue

        cleaned, flags = qc_series(raw["waterlevel_m"])
        raw["waterlevel_raw_m"] = raw["waterlevel_m"]
        raw["waterlevel_m"] = cleaned
        raw["qc_flag"] = flags.values

        out = raw.reset_index()
        out.insert(0, "gauge_id", gid)
        out.insert(1, "water_body", row["water_body"])
        out.insert(2, "regulation_tier", row["regulation_tier"])
        out = build_features(out)
        frames.append(out)

        counts = flags.value_counts()
        qc_rows.append({
            "gauge_id": gid,
            "water_body": row["water_body"],
            "regulation_tier": row["regulation_tier"],
            "n_days": len(raw),
            "pct_complete_raw": round(pct, 2),
            "pct_usable_final": round(100 * out["waterlevel_m"].notna().mean(), 2),
            "n_spike": int(counts.get("spike", 0)),
            "n_flatline": int(counts.get("flatline", 0)),
            "n_interp_short": int(counts.get("interp_short", 0)),
            "n_interp_seasonal": int(counts.get("interp_seasonal", 0)),
            "n_gap_long": int(counts.get("gap_long", 0)),
            "start": str(raw.index.min().date()),
            "end": str(raw.index.max().date()),
        })
        print(f"    {gid} {row['water_body'][:24]:<24} {row['regulation_tier']:<3} "
              f"{pct:5.1f}%  spikes={counts.get('spike', 0):<4} "
              f"flat={counts.get('flatline', 0)}")

    if not frames:
        raise RuntimeError("no lakes passed the completeness filter")
    return pd.concat(frames, ignore_index=True), pd.DataFrame(qc_rows)


# --------------------------------------------------------- 7. data dictionary
def data_dictionary(daily, static):
    defs = {
        "gauge_id": ("BAFU station ID", "-", "CAMELS-CH stations"),
        "water_body": ("Lake name", "-", "CAMELS-CH stations"),
        "regulation_tier": ("N0/N1/R natural classification", "-", "DERIVED from human-influence attributes"),
        "date": ("Observation date", "date", "CAMELS-CH observation_based"),
        "waterlevel_raw_m": ("Water level as published", "m", "CAMELS-CH / BAFU"),
        "waterlevel_m": ("Water level after QC and gap handling", "m", "DERIVED"),
        "qc_flag": ("ok/spike/flatline/interp_short/interp_seasonal/gap_long", "-", "DERIVED"),
        "precipitation_mm": ("Catchment daily precipitation", "mm/d", "CAMELS-CH / MeteoSwiss"),
        "temperature_mean_c": ("Catchment mean air temperature", "degC", "CAMELS-CH / MeteoSwiss"),
        "swe_mm": ("Snow water equivalent", "mm", "CAMELS-CH / SLF"),
        "api_30": ("Antecedent precipitation index, 30d halflife", "mm", "DERIVED"),
        "api_60": ("Antecedent precipitation index, 60d halflife", "mm", "DERIVED"),
        "api_90": ("Antecedent precipitation index, 90d halflife", "mm", "DERIVED"),
        "snowmelt_proxy": ("SWE decrease on days above 0 degC", "mm/d", "DERIVED"),
        "snowmelt_cum30": ("30-day cumulative snowmelt proxy", "mm", "DERIVED"),
        "degree_days": ("Mean temperature floored at zero", "degC", "DERIVED"),
        "cdd_30": ("30-day cumulative degree-days", "degC.d", "DERIVED"),
        "level_delta1": ("Day-on-day level change", "m", "DERIVED"),
        "doy_sin": ("Cyclical day-of-year, sine", "-", "DERIVED"),
        "doy_cos": ("Cyclical day-of-year, cosine", "-", "DERIVED"),
        "specific_storage_mm": ("Upstream reservoir capacity per catchment area", "mm", "DERIVED"),
        "catchment_area_km2": ("Catchment area", "km2", "CAMELS-CH topographic"),
    }
    rows = []
    for table, df in (("lakes_daily", daily), ("lakes_static", static)):
        for c in df.columns:
            if c in defs:
                desc, unit, src = defs[c]
            elif re.match(r"level_lag\d+", c):
                desc, unit, src = (f"Water level lagged {c.split('lag')[1]} days", "m", "DERIVED")
            elif re.match(r"level_roll\d+", c):
                desc, unit, src = (f"Rolling {c} of past levels", "m", "DERIVED")
            else:
                desc, unit, src = ("See CAMELS-CH documentation", "-", "CAMELS-CH")
            rows.append({"table": table, "column": c, "description": desc,
                         "unit": unit, "source": src,
                         "dtype": str(df[c].dtype)})
    return pd.DataFrame(rows)


# ----------------------------------------------------------------- 8. driver
def build(root):
    root = Path(root).expanduser().resolve()
    OUT.mkdir(exist_ok=True)

    lakes = select_lakes(root)
    static = classify_regulation(root, lakes)
    static = attach_attributes(root, static)
    daily, qc = build_panel(root, static)

    static = static[static["gauge_id"].isin(daily["gauge_id"].unique())].reset_index(drop=True)
    static = static.merge(
        qc[["gauge_id", "pct_complete_raw", "pct_usable_final", "start", "end"]],
        on="gauge_id", how="left")

    dd = data_dictionary(daily, static)

    rule("6. WRITING OUTPUTS")
    static.to_csv(OUT / "lakes_static.csv", index=False)
    daily.to_csv(OUT / "lakes_daily.csv", index=False)
    dd.to_csv(OUT / "data_dictionary.csv", index=False)
    qc.to_csv(OUT / "qc_report.csv", index=False)

    xl = OUT / "natural_lakes.xlsx"
    if len(daily) < 1_000_000:
        with pd.ExcelWriter(xl, engine="openpyxl") as w:
            static.to_excel(w, sheet_name="lakes_static", index=False)
            daily.to_excel(w, sheet_name="lakes_daily", index=False)
            qc.to_excel(w, sheet_name="qc_report", index=False)
            dd.to_excel(w, sheet_name="data_dictionary", index=False)
        print(f"    {xl}")
    else:
        print(f"    !! {len(daily):,} rows exceeds the Excel limit; CSVs only")

    for f in ("lakes_static.csv", "lakes_daily.csv", "data_dictionary.csv", "qc_report.csv"):
        print(f"    {OUT / f}")

    rule("SUMMARY")
    print(f"    lakes retained : {static['gauge_id'].nunique()}")
    print(f"    daily rows     : {len(daily):,}")
    print(f"    date range     : {daily['date'].min().date()} to {daily['date'].max().date()}")
    print(f"    features       : {daily.shape[1]} columns")
    print("\n" + static["regulation_tier"].value_counts().to_string())
    print("\n    N0 lakes (your strictly natural primary sample):")
    for _, r in static[static["regulation_tier"] == "N0"].iterrows():
        print(f"      {int(r['gauge_id'])}  {r['water_body']}")
    return static, daily, qc, dd


if __name__ == "__main__":
    if len(sys.argv) > 1:
        build(sys.argv[1])
    else:
        print("usage: python 01_build_dataset.py /path/to/camels_ch")
        print("   or: build(r'C:\\path\\to\\camels_ch')  in a notebook")


1. SELECTING LAKES


FileNotFoundError: CAMELS_CH_gauging_stations.dbf not found

In [4]:
static, daily, qc, dd = build(r"C:\Users\prave\Documents\UoMDS\DISSERTATION\camels_ch")


1. SELECTING LAKES
    331 stations -> 33 Swiss lake stations

2. REGULATION TIERING
    area column:  area
    elev column:  gauge_elevation

regulation_tier
N1    18
N0    15

  gauge_id          water_body regulation_tier  hp_count  num_reservoir  specific_storage_mm
     2004           Murtensee              N0         0              0                0.000
     2168        Sempachersee              N0         0              0                0.000
     2137        Baldeggersee              N0         0              0                0.000
     2101      Lago_di_Lugano              N0         0              0                0.000
     2097        Hallwilersee              N0         0              0                0.000
     2082          Greifensee              N0         0              0                0.000
     2081        Pfäffikersee              N0         0              0                0.000
     2484         Lauerzersee              N0         0              0              

## H


In [6]:
"""
 RQ1: how do meteorological drivers lag into lake level,
and does lake elevation modulate that lag?

Notebook:
    import pandas as pd
    daily  = pd.read_csv("output/lakes_daily.csv", parse_dates=["date"])
    static = pd.read_csv("output/lakes_static.csv")
    res = run_lag_analysis(daily, static, sample="N0")

Outputs (into ./output and ./figures):
    ccf_results.csv         full cross-correlation surface, every lake x driver x lag
    peak_lags.csv           peak-correlation lag per lake per driver
    granger_results.csv     Granger causality p-values
    elevation_regression.txt  OLS of peak lag on lake attributes
    duplicate_check.csv     correlation between co-located stations
    figures/*.png

"""

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT = Path("output")
FIG = Path("figures")

MAX_LAG = 60          # days of lag to scan
DRIVERS = ["precipitation_mm", "temperature_mean_c", "snowmelt_proxy"]
TARGET = "waterlevel_m"


def rule(t):
    print(f"\n{'=' * 78}\n{t}\n{'=' * 78}")


# --------------------------------------------------------- sample construction
def deduplicate(daily, static, sample="N0"):
    """
    One station per water body. Two gauges on the same lake are not independent
    observations - pooling them is pseudo-replication.
    """
    rule(f"SAMPLE CONSTRUCTION (tier filter: {sample})")

    st = static.copy()
    if sample != "ALL":
        st = st[st["regulation_tier"].isin(list(sample) if isinstance(sample, list) else [sample])]

    completeness = "pct_usable_final" if "pct_usable_final" in st.columns else "pct_complete_raw"
    st = st.sort_values(completeness, ascending=False)
    keep = st.drop_duplicates(subset="water_body", keep="first")

    dropped = st[~st["gauge_id"].isin(keep["gauge_id"])]
    if len(dropped):
        print("    dropped as duplicate water bodies:")
        for _, r in dropped.iterrows():
            print(f"      {int(r['gauge_id'])}  {r['water_body']}")

    print(f"\n    {len(st)} stations -> {len(keep)} distinct water bodies")
    d = daily[daily["gauge_id"].isin(keep["gauge_id"])].copy()
    return d, keep.reset_index(drop=True)


def duplicate_check(daily, static):
    """Co-located stations should track each other. If they don't, suspect the datum."""
    rule("DUPLICATE STATION CHECK")
    rows = []
    for wb, grp in static.groupby("water_body"):
        if len(grp) < 2:
            continue
        ids = grp["gauge_id"].tolist()
        for i in range(len(ids)):
            for j in range(i + 1, len(ids)):
                a = daily[daily["gauge_id"] == ids[i]].set_index("date")[TARGET]
                b = daily[daily["gauge_id"] == ids[j]].set_index("date")[TARGET]
                joined = pd.concat([a, b], axis=1, join="inner").dropna()
                if len(joined) < 100:
                    continue
                r = joined.iloc[:, 0].corr(joined.iloc[:, 1])
                dr = joined.iloc[:, 0].diff().corr(joined.iloc[:, 1].diff())
                rows.append({"water_body": wb, "gauge_a": ids[i], "gauge_b": ids[j],
                             "n_days": len(joined), "corr_level": round(r, 4),
                             "corr_daily_change": round(dr, 4),
                             "mean_offset_m": round(
                                 (joined.iloc[:, 0] - joined.iloc[:, 1]).mean(), 3)})
    if not rows:
        print("    no co-located stations in this sample")
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    print("\n    corr_level near 1 = same signal. A large mean_offset_m with high")
    print("    correlation means different gauge datums, not different behaviour.")
    df.to_csv(OUT / "duplicate_check.csv", index=False)
    return df


# ------------------------------------------------------------ seasonal anomaly
def deseasonalise(s, dates):
    """
    Remove the day-of-year climatology.

    raw lake level and raw precipitation both carry a strong annual
    cycle, so their cross-correlation mostly measures 'both are seasonal' rather
    than any causal lag. Working on anomalies is what makes the CCF interpretable.
    """
    doy = dates.dt.dayofyear
    clim = s.groupby(doy).transform("mean")
    return s - clim


# ------------------------------------------------------------------------ CCF
def cross_correlation(target, driver, max_lag=MAX_LAG):
    """Correlate driver at t-k against target at t, for k = 0..max_lag."""
    out = []
    for k in range(max_lag + 1):
        d = driver.shift(k)
        pair = pd.concat([target, d], axis=1).dropna()
        if len(pair) < 200:
            out.append(np.nan)
            continue
        out.append(pair.iloc[:, 0].corr(pair.iloc[:, 1]))
    return np.array(out)


def run_ccf(daily, static):
    rule("CROSS-CORRELATION: DRIVERS vs LEVEL")
    ccf_rows, peak_rows = [], []

    for _, meta in static.iterrows():
        gid = int(meta["gauge_id"])
        g = daily[daily["gauge_id"] == gid].sort_values("date").copy()
        if len(g) < 1000:
            continue

        lvl_a = deseasonalise(g[TARGET], g["date"])
        for drv in DRIVERS:
            if drv not in g.columns or g[drv].notna().sum() < 1000:
                continue
            drv_a = deseasonalise(g[drv], g["date"])
            ccf = cross_correlation(lvl_a.reset_index(drop=True),
                                    drv_a.reset_index(drop=True))
            if np.all(np.isnan(ccf)):
                continue

            for k, v in enumerate(ccf):
                ccf_rows.append({"gauge_id": gid, "water_body": meta["water_body"],
                                 "driver": drv, "lag_days": k, "corr": v})

            peak_k = int(np.nanargmax(np.abs(ccf)))
            peak_rows.append({
                "gauge_id": gid,
                "water_body": meta["water_body"],
                "regulation_tier": meta.get("regulation_tier"),
                "driver": drv,
                "peak_lag_days": peak_k,
                "peak_corr": round(float(ccf[peak_k]), 4),
                "corr_at_lag0": round(float(ccf[0]), 4),
            })

    ccf_df = pd.DataFrame(ccf_rows)
    peak_df = pd.DataFrame(peak_rows)
    ccf_df.to_csv(OUT / "ccf_results.csv", index=False)
    peak_df.to_csv(OUT / "peak_lags.csv", index=False)

    for drv in DRIVERS:
        sub = peak_df[peak_df["driver"] == drv]
        if len(sub):
            print(f"\n    {drv}")
            print(sub[["water_body", "peak_lag_days", "peak_corr"]]
                  .sort_values("peak_lag_days").to_string(index=False))
    return ccf_df, peak_df


# -------------------------------------------------------------------- Granger
def run_granger(daily, static, maxlag=30):
    rule("GRANGER CAUSALITY (on stationary first differences)")
    try:
        from statsmodels.tsa.stattools import grangercausalitytests, adfuller
    except ImportError:
        print("    statsmodels not installed - skipping")
        return pd.DataFrame()

    rows = []
    for _, meta in static.iterrows():
        gid = int(meta["gauge_id"])
        g = daily[daily["gauge_id"] == gid].sort_values("date")
        for drv in DRIVERS:
            if drv not in g.columns:
                continue
            pair = pd.DataFrame({
                "y": g[TARGET].diff(),
                "x": g[drv].diff(),
            }).dropna()
            if len(pair) < 500 or pair["x"].std() == 0:
                continue
            try:
                adf_p = adfuller(pair["y"], autolag="AIC")[1]
                res = grangercausalitytests(pair[["y", "x"]], maxlag=maxlag, verbose=False)
                pvals = {k: v[0]["ssr_ftest"][1] for k, v in res.items()}
                best = min(pvals, key=pvals.get)
                rows.append({"gauge_id": gid, "water_body": meta["water_body"],
                             "driver": drv, "adf_p_target": round(adf_p, 5),
                             "best_lag": best, "min_p": pvals[best],
                             "significant_5pct": pvals[best] < 0.05})
            except Exception as e:
                print(f"    {gid} {drv}: {e}")

    df = pd.DataFrame(rows)
    if len(df):
        df.to_csv(OUT / "granger_results.csv", index=False)
        print(df.to_string(index=False))
        print("\n    Note: Granger causality is predictive precedence, not physical")
        print("    causation. Phrase it that way in the write-up.")
    return df


# --------------------------------------------------- does elevation drive lag?
def elevation_regression(peak_df, static):
    rule("PEAK LAG vs LAKE ATTRIBUTES")
    try:
        import statsmodels.api as sm
    except ImportError:
        print("    statsmodels not installed - skipping")
        return

    candidates = ["gauge_elevation", "area", "catchment_area_km2", "frac_snow",
                  "glac_area", "p_mean", "aridity", "mean_slope"]
    have = [c for c in candidates if c in static.columns]
    print(f"    predictors available: {have}")

    lines = []
    for drv in DRIVERS:
        sub = peak_df[peak_df["driver"] == drv].merge(
            static[["gauge_id"] + have], on="gauge_id", how="left")
        sub = sub.dropna(subset=["peak_lag_days"])
        if len(sub) < 6:
            continue

        preds = [c for c in have if sub[c].notna().sum() >= len(sub) - 1
                 and sub[c].std(skipna=True) > 0]
        # with ~14 lakes keep the model small: 3 predictors maximum
        preds = preds[:3]
        if not preds:
            continue

        X = sm.add_constant(sub[preds].astype(float).fillna(sub[preds].astype(float).mean()))
        y = sub["peak_lag_days"].astype(float)
        model = sm.OLS(y, X).fit()

        header = f"\n### peak lag of {drv} ~ {' + '.join(preds)}   (n={len(sub)})"
        print(header)
        print(model.summary().as_text())
        lines.append(header + "\n" + model.summary().as_text())

        # simple bivariate correlation is more honest at this n
        if "gauge_elevation" in sub.columns:
            r = sub["peak_lag_days"].corr(sub["gauge_elevation"].astype(float))
            note = f"    Pearson r(peak_lag, elevation) = {r:.3f}"
            print(note)
            lines.append(note)

    (OUT / "elevation_regression.txt").write_text("\n\n".join(lines), encoding="utf-8")
    print(f"\n    -> {OUT / 'elevation_regression.txt'}")
    print("    With n around 14, treat multivariate coefficients as descriptive.")
    print("    The bivariate correlation is the defensible headline.")


# ------------------------------------------------------------------- figures
def make_figures(ccf_df, peak_df, static, daily):
    rule("FIGURES")
    FIG.mkdir(exist_ok=True)

    elev = static.set_index("gauge_id").get("gauge_elevation", pd.Series(dtype=float))

    # Fig 1: CCF curves, coloured by elevation
    for drv in DRIVERS:
        sub = ccf_df[ccf_df["driver"] == drv]
        if sub.empty:
            continue
        fig, ax = plt.subplots(figsize=(9, 5.5))
        gids = sub["gauge_id"].unique()
        vals = [elev.get(g, np.nan) for g in gids]
        lo, hi = np.nanmin(vals), np.nanmax(vals)
        for gid in gids:
            s = sub[sub["gauge_id"] == gid]
            e = elev.get(gid, np.nan)
            shade = 0.0 if not np.isfinite(e) or hi == lo else (e - lo) / (hi - lo)
            ax.plot(s["lag_days"], s["corr"], color=plt.cm.viridis(shade),
                    lw=1.4, alpha=0.85,
                    label=f"{s['water_body'].iloc[0]} ({e:.0f} m)"
                    if np.isfinite(e) else s["water_body"].iloc[0])
        ax.axhline(0, color="k", lw=0.6)
        ax.set_xlabel("Lag (days)")
        ax.set_ylabel("Correlation (seasonal anomalies)")
        ax.set_title(f"Cross-correlation: {drv} leading lake level")
        ax.legend(fontsize=6.5, ncol=2, frameon=False)
        fig.tight_layout()
        fig.savefig(FIG / f"ccf_{drv}.png", dpi=180)
        plt.close(fig)
        print(f"    figures/ccf_{drv}.png")

    # Fig 2: peak lag against elevation
    if len(elev):
        fig, axes = plt.subplots(1, len(DRIVERS), figsize=(4.2 * len(DRIVERS), 4), sharey=True)
        axes = np.atleast_1d(axes)
        for ax, drv in zip(axes, DRIVERS):
            sub = peak_df[peak_df["driver"] == drv]
            if sub.empty:
                continue
            x = [elev.get(g, np.nan) for g in sub["gauge_id"]]
            ax.scatter(x, sub["peak_lag_days"], s=45, alpha=0.8, edgecolor="k", lw=0.5)
            for xi, yi, nm in zip(x, sub["peak_lag_days"], sub["water_body"]):
                if np.isfinite(xi):
                    ax.annotate(nm[:11], (xi, yi), fontsize=5.5,
                                xytext=(3, 3), textcoords="offset points")
            ax.set_xlabel("Lake elevation (m)")
            ax.set_title(drv, fontsize=9)
        axes[0].set_ylabel("Peak-correlation lag (days)")
        fig.tight_layout()
        fig.savefig(FIG / "peak_lag_vs_elevation.png", dpi=180)
        plt.close(fig)
        print("    figures/peak_lag_vs_elevation.png")

    # Fig 3: annual cycle per lake, standardised so lakes are comparable
    fig, ax = plt.subplots(figsize=(9, 5))
    for gid, g in daily.groupby("gauge_id"):
        s = g.set_index("date")[TARGET]
        z = (s - s.mean()) / s.std()
        clim = z.groupby(z.index.dayofyear).mean()
        e = elev.get(gid, np.nan)
        lo, hi = np.nanmin(elev), np.nanmax(elev)
        shade = 0.0 if not np.isfinite(e) or hi == lo else (e - lo) / (hi - lo)
        ax.plot(clim.index, clim.values, lw=1.3, alpha=0.85, color=plt.cm.viridis(shade))
    ax.set_xlabel("Day of year")
    ax.set_ylabel("Standardised level anomaly")
    ax.set_title("Annual cycle by lake (dark = low elevation, bright = high)")
    fig.tight_layout()
    fig.savefig(FIG / "annual_cycle.png", dpi=180)
    plt.close(fig)
    print("    figures/annual_cycle.png")


# -------------------------------------------------------------------- driver
def run_lag_analysis(daily, static, sample="N0"):
    OUT.mkdir(exist_ok=True)
    FIG.mkdir(exist_ok=True)

    duplicate_check(daily, static)
    d, st = deduplicate(daily, static, sample=sample)
    ccf_df, peak_df = run_ccf(d, st)
    granger_df = run_granger(d, st)
    elevation_regression(peak_df, st)
    make_figures(ccf_df, peak_df, st, d)

    rule("DONE - RQ1 EVIDENCE COMPLETE")
    
    return {"daily": d, "static": st, "ccf": ccf_df,
            "peaks": peak_df, "granger": granger_df}

In [7]:
import pandas as pd
daily  = pd.read_csv("output/lakes_daily.csv", parse_dates=["date"])
static = pd.read_csv("output/lakes_static.csv")

res = run_lag_analysis(daily, static, sample="N0")        # primary, 14 lakes
res_all = run_lag_analysis(daily, static, sample="ALL")   # sensitivity, all tiers


DUPLICATE STATION CHECK
        water_body  gauge_a  gauge_b  n_days  corr_level  corr_daily_change  mean_offset_m
         Lac_Léman     2027     2028   14610      0.9981             0.8801          0.003
  Lac_de_Neuchâtel     2154     2642   14610      0.9938             0.8397          0.001
     Lago_Maggiore     2022     2074   14610      0.9980             0.9553          0.002
    Lago_di_Lugano     2021     2101   14608      0.9835             0.8550         -0.009
Vierwaldstättersee     2025     2207   14610      0.9940             0.9344         -0.005

    corr_level near 1 = same signal. A large mean_offset_m with high
    correlation means different gauge datums, not different behaviour.

SAMPLE CONSTRUCTION (tier filter: N0)
    dropped as duplicate water bodies:
      2021  Lago_di_Lugano

    15 stations -> 14 distinct water bodies

CROSS-CORRELATION: DRIVERS vs LEVEL

    precipitation_mm
     water_body  peak_lag_days  peak_corr
St._Moritzersee              1     0.

C:\Users\prave\anaconda3\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
C:\Users\prave\anaconda3\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
C:\Users\prave\anaconda3\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
C:\Users\prave\anaconda3\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
C:\Users\prave\anaconda3\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
C:\Users\prave\anaconda3\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions shou

 gauge_id      water_body             driver  adf_p_target  best_lag         min_p  significant_5pct
     2004       Murtensee   precipitation_mm           0.0         1  0.000000e+00              True
     2004       Murtensee temperature_mean_c           0.0        20  1.814048e-46              True
     2004       Murtensee     snowmelt_proxy           0.0         4  8.744558e-23              True
     2017        Zugersee   precipitation_mm           0.0         1  0.000000e+00              True
     2017        Zugersee temperature_mean_c           0.0         4  7.063161e-28              True
     2017        Zugersee     snowmelt_proxy           0.0         7  4.946283e-07              True
     2031        Ägerisee   precipitation_mm           0.0         1  0.000000e+00              True
     2031        Ägerisee temperature_mean_c           0.0         4  5.135109e-29              True
     2031        Ägerisee     snowmelt_proxy           0.0        11  1.098125e-17         

C:\Users\prave\anaconda3\Lib\site-packages\scipy\stats\_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=14 observations were given.
  return hypotest_fun_in(*args, **kwds)
C:\Users\prave\anaconda3\Lib\site-packages\scipy\stats\_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=14 observations were given.
  return hypotest_fun_in(*args, **kwds)
C:\Users\prave\anaconda3\Lib\site-packages\scipy\stats\_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=14 observations were given.
  return hypotest_fun_in(*args, **kwds)
C:\Users\prave\anaconda3\Lib\site-packages\scipy\stats\_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=14 observations were given.
  return hypotest_fun_in(*args, **kwds)
C:\Users\prave\anaconda3\Lib\sit

    figures/ccf_precipitation_mm.png
    figures/ccf_temperature_mean_c.png
    figures/ccf_snowmelt_proxy.png


ValueError: zero-size array to reduction operation minimum which has no identity

In [8]:
"""
 RQ1: how do meteorological drivers lag into lake level,
and does lake elevation modulate that lag?

Notebook:
    import pandas as pd
    daily  = pd.read_csv("output/lakes_daily.csv", parse_dates=["date"])
    static = pd.read_csv("output/lakes_static.csv")
    res = run_lag_analysis(daily, static, sample="N0")

Outputs (into ./output and ./figures):
    ccf_results.csv         full cross-correlation surface, every lake x driver x lag
    peak_lags.csv           peak-correlation lag per lake per driver
    granger_results.csv     Granger causality p-values
    elevation_regression.txt  OLS of peak lag on lake attributes
    duplicate_check.csv     correlation between co-located stations
    figures/*.png

Dependencies: pandas, numpy, scipy, statsmodels, matplotlib.
"""

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT = Path("output")
FIG = Path("figures")

MAX_LAG = 60          # days of lag to scan
DRIVERS = ["precipitation_mm", "temperature_mean_c", "snowmelt_proxy"]
TARGET = "waterlevel_m"

# 01_build_dataset renames the topographic elevation column, so accept either name.
ELEV_NAMES = ["mean_elevation_m", "gauge_elevation", "elev_mean", "elevation"]


def elevation_series(static):
    """Return gauge_id -> elevation, whatever the column ended up being called."""
    for c in ELEV_NAMES:
        if c in static.columns and static[c].notna().any():
            return pd.to_numeric(static.set_index("gauge_id")[c], errors="coerce"), c
    for c in static.columns:
        if "elev" in c.lower() and static[c].notna().any():
            return pd.to_numeric(static.set_index("gauge_id")[c], errors="coerce"), c
    return pd.Series(dtype=float), None


def rule(t):
    print(f"\n{'=' * 78}\n{t}\n{'=' * 78}")


# --------------------------------------------------------- sample construction
def deduplicate(daily, static, sample="N0"):
    """
    One station per water body. Two gauges on the same lake are not independent
    observations - pooling them is pseudo-replication.
    """
    rule(f"SAMPLE CONSTRUCTION (tier filter: {sample})")

    st = static.copy()
    if sample != "ALL":
        st = st[st["regulation_tier"].isin(list(sample) if isinstance(sample, list) else [sample])]

    completeness = "pct_usable_final" if "pct_usable_final" in st.columns else "pct_complete_raw"
    st = st.sort_values(completeness, ascending=False)
    keep = st.drop_duplicates(subset="water_body", keep="first")

    dropped = st[~st["gauge_id"].isin(keep["gauge_id"])]
    if len(dropped):
        print("    dropped as duplicate water bodies:")
        for _, r in dropped.iterrows():
            print(f"      {int(r['gauge_id'])}  {r['water_body']}")

    print(f"\n    {len(st)} stations -> {len(keep)} distinct water bodies")
    d = daily[daily["gauge_id"].isin(keep["gauge_id"])].copy()
    return d, keep.reset_index(drop=True)


def duplicate_check(daily, static):
    """Co-located stations should track each other. If they don't, suspect the datum."""
    rule("DUPLICATE STATION CHECK")
    rows = []
    for wb, grp in static.groupby("water_body"):
        if len(grp) < 2:
            continue
        ids = grp["gauge_id"].tolist()
        for i in range(len(ids)):
            for j in range(i + 1, len(ids)):
                a = daily[daily["gauge_id"] == ids[i]].set_index("date")[TARGET]
                b = daily[daily["gauge_id"] == ids[j]].set_index("date")[TARGET]
                joined = pd.concat([a, b], axis=1, join="inner").dropna()
                if len(joined) < 100:
                    continue
                r = joined.iloc[:, 0].corr(joined.iloc[:, 1])
                dr = joined.iloc[:, 0].diff().corr(joined.iloc[:, 1].diff())
                rows.append({"water_body": wb, "gauge_a": ids[i], "gauge_b": ids[j],
                             "n_days": len(joined), "corr_level": round(r, 4),
                             "corr_daily_change": round(dr, 4),
                             "mean_offset_m": round(
                                 (joined.iloc[:, 0] - joined.iloc[:, 1]).mean(), 3)})
    if not rows:
        print("    no co-located stations in this sample")
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    print("\n    corr_level near 1 = same signal. A large mean_offset_m with high")
    print("    correlation means different gauge datums, not different behaviour.")
    df.to_csv(OUT / "duplicate_check.csv", index=False)
    return df


# ------------------------------------------------------------ seasonal anomaly
def deseasonalise(s, dates):
    """
    Remove the day-of-year climatology.

    raw lake level and raw precipitation both carry a strong annual
    cycle, so their cross-correlation mostly measures 'both are seasonal' rather
    than any causal lag. Working on anomalies is what makes the CCF interpretable.
    """
    doy = dates.dt.dayofyear
    clim = s.groupby(doy).transform("mean")
    return s - clim


# ------------------------------------------------------------------------ CCF
def cross_correlation(target, driver, max_lag=MAX_LAG):
    """Correlate driver at t-k against target at t, for k = 0..max_lag."""
    out = []
    for k in range(max_lag + 1):
        d = driver.shift(k)
        pair = pd.concat([target, d], axis=1).dropna()
        if len(pair) < 200:
            out.append(np.nan)
            continue
        out.append(pair.iloc[:, 0].corr(pair.iloc[:, 1]))
    return np.array(out)


def run_ccf(daily, static):
    rule("CROSS-CORRELATION: DRIVERS vs LEVEL")
    ccf_rows, peak_rows = [], []

    for _, meta in static.iterrows():
        gid = int(meta["gauge_id"])
        g = daily[daily["gauge_id"] == gid].sort_values("date").copy()
        if len(g) < 1000:
            continue

        lvl_a = deseasonalise(g[TARGET], g["date"])
        for drv in DRIVERS:
            if drv not in g.columns or g[drv].notna().sum() < 1000:
                continue
            drv_a = deseasonalise(g[drv], g["date"])
            ccf = cross_correlation(lvl_a.reset_index(drop=True),
                                    drv_a.reset_index(drop=True))
            if np.all(np.isnan(ccf)):
                continue

            for k, v in enumerate(ccf):
                ccf_rows.append({"gauge_id": gid, "water_body": meta["water_body"],
                                 "driver": drv, "lag_days": k, "corr": v})

            peak_k = int(np.nanargmax(np.abs(ccf)))
            peak_rows.append({
                "gauge_id": gid,
                "water_body": meta["water_body"],
                "regulation_tier": meta.get("regulation_tier"),
                "driver": drv,
                "peak_lag_days": peak_k,
                "peak_corr": round(float(ccf[peak_k]), 4),
                "corr_at_lag0": round(float(ccf[0]), 4),
                "peak_sign": "positive" if ccf[peak_k] > 0 else "negative",
                # a peak within 5 days of the scan limit may be truncated:
                # the true maximum could lie beyond MAX_LAG
                "peak_at_boundary": peak_k >= MAX_LAG - 5,
            })

    ccf_df = pd.DataFrame(ccf_rows)
    peak_df = pd.DataFrame(peak_rows)
    ccf_df.to_csv(OUT / "ccf_results.csv", index=False)
    peak_df.to_csv(OUT / "peak_lags.csv", index=False)

    for drv in DRIVERS:
        sub = peak_df[peak_df["driver"] == drv]
        if len(sub):
            print(f"\n    {drv}")
            print(sub[["water_body", "peak_lag_days", "peak_corr", "peak_sign",
                       "peak_at_boundary"]]
                  .sort_values("peak_lag_days").to_string(index=False))
            edge = sub[sub["peak_at_boundary"]]
            if len(edge):
                print(f"    !! peak at scan limit for: {list(edge['water_body'])}")
                print(f"       True peak may exceed {MAX_LAG} days - say so, or raise MAX_LAG.")
    return ccf_df, peak_df


# -------------------------------------------------------------------- Granger
def run_granger(daily, static, maxlag=30):
    rule("GRANGER CAUSALITY (on stationary first differences)")
    try:
        from statsmodels.tsa.stattools import grangercausalitytests, adfuller
    except ImportError:
        print("    statsmodels not installed - skipping")
        return pd.DataFrame()

    rows = []
    for _, meta in static.iterrows():
        gid = int(meta["gauge_id"])
        g = daily[daily["gauge_id"] == gid].sort_values("date")
        for drv in DRIVERS:
            if drv not in g.columns:
                continue
            pair = pd.DataFrame({
                "y": g[TARGET].diff(),
                "x": g[drv].diff(),
            }).dropna()
            if len(pair) < 500 or pair["x"].std() == 0:
                continue
            try:
                adf_p = adfuller(pair["y"], autolag="AIC")[1]
                res = grangercausalitytests(pair[["y", "x"]], maxlag=maxlag)
                pvals = {k: v[0]["ssr_ftest"][1] for k, v in res.items()}
                best = min(pvals, key=pvals.get)
                rows.append({"gauge_id": gid, "water_body": meta["water_body"],
                             "driver": drv, "adf_p_target": round(adf_p, 5),
                             "best_lag": best, "min_p": pvals[best],
                             "significant_5pct": pvals[best] < 0.05})
            except Exception as e:
                print(f"    {gid} {drv}: {e}")

    df = pd.DataFrame(rows)
    if len(df):
        df.to_csv(OUT / "granger_results.csv", index=False)
        print(df.to_string(index=False))
        print("\n    Note: Granger causality is predictive precedence, not physical")
        print("    causation. Phrase it that way in the write-up.")
    return df


# --------------------------------------------------- does elevation drive lag?
def elevation_regression(peak_df, static):
    rule("PEAK LAG vs LAKE ATTRIBUTES")
    try:
        import statsmodels.api as sm
    except ImportError:
        print("    statsmodels not installed - skipping")
        return

    elev, elev_col = elevation_series(static)
    if elev_col:
        print(f"    elevation column resolved to: {elev_col}")
    else:
        print("    !! no elevation column found - the headline test cannot run")

    candidates = ELEV_NAMES + ["area", "catchment_area_km2", "frac_snow",
                               "glac_area", "p_mean", "aridity", "mean_slope"]
    have = [c for c in dict.fromkeys(candidates) if c in static.columns
            and static[c].notna().any()]
    print(f"    predictors available: {have}")

    lines = []
    for drv in DRIVERS:
        sub = peak_df[peak_df["driver"] == drv].merge(
            static[["gauge_id"] + have], on="gauge_id", how="left")
        sub = sub.dropna(subset=["peak_lag_days"])
        if len(sub) < 6:
            continue

        # Bivariate first - at n=14 this is the defensible result.
        header = f"\n### {drv}: peak lag vs each attribute   (n={len(sub)})"
        print(header)
        lines.append(header)
        for c in have:
            x = pd.to_numeric(sub[c], errors="coerce")
            if x.notna().sum() < 6 or x.std(skipna=True) == 0:
                continue
            r = sub["peak_lag_days"].astype(float).corr(x)
            rs = sub["peak_lag_days"].astype(float).corr(x, method="spearman")
            line = f"    r(peak_lag, {c:<20}) = {r:+.3f}   spearman = {rs:+.3f}"
            print(line)
            lines.append(line)

        # Sign of the peak correlation against elevation - the mechanism test.
        if elev_col and "peak_sign" in sub.columns:
            sub["_elev"] = sub["gauge_id"].map(elev)
            pos = sub[sub["peak_sign"] == "positive"]["_elev"].dropna()
            neg = sub[sub["peak_sign"] == "negative"]["_elev"].dropna()
            if len(pos) and len(neg):
                msg = (f"    peak correlation POSITIVE at mean elevation {pos.mean():.0f} m "
                       f"(n={len(pos)}), NEGATIVE at {neg.mean():.0f} m (n={len(neg)})")
                print(msg)
                lines.append(msg)

        preds = [c for c in have if pd.to_numeric(sub[c], errors="coerce").notna().sum()
                 >= len(sub) - 1 and pd.to_numeric(sub[c], errors="coerce").std() > 0]
        if elev_col and elev_col in preds:                 # keep elevation, drop others
            preds = [elev_col] + [p for p in preds if p != elev_col]
        preds = preds[:3]
        if not preds:
            continue

        Xv = sub[preds].apply(pd.to_numeric, errors="coerce")
        X = sm.add_constant(Xv.fillna(Xv.mean()))
        y = sub["peak_lag_days"].astype(float)
        model = sm.OLS(y, X).fit()
        txt = f"\n### OLS: peak lag of {drv} ~ {' + '.join(preds)}   (n={len(sub)})"
        print(txt)
        print(model.summary().as_text())
        lines.append(txt + "\n" + model.summary().as_text())

    (OUT / "elevation_regression.txt").write_text("\n\n".join(lines), encoding="utf-8")
    print(f"\n    -> {OUT / 'elevation_regression.txt'}")
    print("    With n around 14, treat multivariate coefficients as descriptive.")
    print("    The bivariate correlation is the defensible headline.")


# ------------------------------------------------------------------- figures
def make_figures(ccf_df, peak_df, static, daily):
    rule("FIGURES")
    FIG.mkdir(exist_ok=True)

    elev, elev_col = elevation_series(static)
    if len(elev.dropna()):
        lo, hi = float(np.nanmin(elev)), float(np.nanmax(elev))
    else:
        lo = hi = np.nan

    def shade_for(gid):
        e = elev.get(gid, np.nan)
        if not np.isfinite(e) or not np.isfinite(lo) or hi == lo:
            return 0.5, e
        return (e - lo) / (hi - lo), e

    # Fig 1: CCF curves, coloured by elevation
    for drv in DRIVERS:
        sub = ccf_df[ccf_df["driver"] == drv]
        if sub.empty:
            continue
        fig, ax = plt.subplots(figsize=(9, 5.5))
        for gid in sub["gauge_id"].unique():
            s = sub[sub["gauge_id"] == gid]
            shade, e = shade_for(gid)
            ax.plot(s["lag_days"], s["corr"], color=plt.cm.viridis(shade),
                    lw=1.4, alpha=0.85,
                    label=f"{s['water_body'].iloc[0]} ({e:.0f} m)"
                    if np.isfinite(e) else s["water_body"].iloc[0])
        ax.axhline(0, color="k", lw=0.6)
        ax.set_xlabel("Lag (days)")
        ax.set_ylabel("Correlation (seasonal anomalies)")
        ax.set_title(f"Cross-correlation: {drv} leading lake level")
        ax.legend(fontsize=6.5, ncol=2, frameon=False)
        fig.tight_layout()
        fig.savefig(FIG / f"ccf_{drv}.png", dpi=180)
        plt.close(fig)
        print(f"    figures/ccf_{drv}.png")

    # Fig 2: peak lag against elevation
    if len(elev):
        fig, axes = plt.subplots(1, len(DRIVERS), figsize=(4.2 * len(DRIVERS), 4), sharey=True)
        axes = np.atleast_1d(axes)
        for ax, drv in zip(axes, DRIVERS):
            sub = peak_df[peak_df["driver"] == drv]
            if sub.empty:
                continue
            x = [elev.get(g, np.nan) for g in sub["gauge_id"]]
            ax.scatter(x, sub["peak_lag_days"], s=45, alpha=0.8, edgecolor="k", lw=0.5)
            for xi, yi, nm in zip(x, sub["peak_lag_days"], sub["water_body"]):
                if np.isfinite(xi):
                    ax.annotate(nm[:11], (xi, yi), fontsize=5.5,
                                xytext=(3, 3), textcoords="offset points")
            ax.set_xlabel("Lake elevation (m)")
            ax.set_title(drv, fontsize=9)
        axes[0].set_ylabel("Peak-correlation lag (days)")
        fig.tight_layout()
        fig.savefig(FIG / "peak_lag_vs_elevation.png", dpi=180)
        plt.close(fig)
        print("    figures/peak_lag_vs_elevation.png")

    # Fig 3: annual cycle per lake, standardised so lakes are comparable
    fig, ax = plt.subplots(figsize=(9, 5))
    for gid, g in daily.groupby("gauge_id"):
        s = g.set_index("date")[TARGET]
        z = (s - s.mean()) / s.std()
        clim = z.groupby(z.index.dayofyear).mean()
        shade, _ = shade_for(gid)
        ax.plot(clim.index, clim.values, lw=1.3, alpha=0.85, color=plt.cm.viridis(shade))
    ax.set_xlabel("Day of year")
    ax.set_ylabel("Standardised level anomaly")
    ax.set_title("Annual cycle by lake (dark = low elevation, bright = high)")
    fig.tight_layout()
    fig.savefig(FIG / "annual_cycle.png", dpi=180)
    plt.close(fig)
    print("    figures/annual_cycle.png")


# -------------------------------------------------------------------- driver
def run_lag_analysis(daily, static, sample="N0"):
    OUT.mkdir(exist_ok=True)
    FIG.mkdir(exist_ok=True)

    duplicate_check(daily, static)
    d, st = deduplicate(daily, static, sample=sample)
    ccf_df, peak_df = run_ccf(d, st)
    granger_df = run_granger(d, st)
    elevation_regression(peak_df, st)
    make_figures(ccf_df, peak_df, st, d)

    rule("DONE - RQ1 EVIDENCE COMPLETE")
    print("    Write this section up now, before starting the models.")
    return {"daily": d, "static": st, "ccf": ccf_df,
            "peaks": peak_df, "granger": granger_df}

In [10]:
"""
RQ1: how do meteorological drivers lag into lake level,
and does lake elevation modulate that lag?

Notebook:
    import pandas as pd
    daily  = pd.read_csv("output/lakes_daily.csv", parse_dates=["date"])
    static = pd.read_csv("output/lakes_static.csv")
    res = run_lag_analysis(daily, static, sample="N0")

Outputs (into ./output and ./figures):
    ccf_results.csv         full cross-correlation surface, every lake x driver x lag
    peak_lags.csv           peak-correlation lag per lake per driver
    granger_results.csv     Granger causality p-values
    elevation_regression.txt  OLS of peak lag on lake attributes
    duplicate_check.csv     correlation between co-located stations
    figures/*.png

Dependencies: pandas, numpy, scipy, statsmodels, matplotlib.
"""

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT = Path("output")
FIG = Path("figures")

MAX_LAG = 60          # days of lag to scan
DRIVERS = ["precipitation_mm", "temperature_mean_c", "snowmelt_proxy"]
TARGET = "waterlevel_m"

# 01_build_dataset renames the topographic elevation column, so accept either name.
ELEV_NAMES = ["mean_elevation_m", "gauge_elevation", "elev_mean", "elevation"]


def elevation_series(static):
    """Return gauge_id -> elevation, whatever the column ended up being called."""
    for c in ELEV_NAMES:
        if c in static.columns and static[c].notna().any():
            return pd.to_numeric(static.set_index("gauge_id")[c], errors="coerce"), c
    for c in static.columns:
        if "elev" in c.lower() and static[c].notna().any():
            return pd.to_numeric(static.set_index("gauge_id")[c], errors="coerce"), c
    return pd.Series(dtype=float), None


def rule(t):
    print(f"\n{'=' * 78}\n{t}\n{'=' * 78}")


# --------------------------------------------------------- sample construction
def deduplicate(daily, static, sample="N0"):
    """
    One station per water body. Two gauges on the same lake are not independent
    observations - pooling them is pseudo-replication.
    """
    rule(f"SAMPLE CONSTRUCTION (tier filter: {sample})")

    st = static.copy()
    if sample != "ALL":
        st = st[st["regulation_tier"].isin(list(sample) if isinstance(sample, list) else [sample])]

    completeness = "pct_usable_final" if "pct_usable_final" in st.columns else "pct_complete_raw"
    st = st.sort_values(completeness, ascending=False)
    keep = st.drop_duplicates(subset="water_body", keep="first")

    dropped = st[~st["gauge_id"].isin(keep["gauge_id"])]
    if len(dropped):
        print("    dropped as duplicate water bodies:")
        for _, r in dropped.iterrows():
            print(f"      {int(r['gauge_id'])}  {r['water_body']}")

    print(f"\n    {len(st)} stations -> {len(keep)} distinct water bodies")
    d = daily[daily["gauge_id"].isin(keep["gauge_id"])].copy()
    return d, keep.reset_index(drop=True)


def duplicate_check(daily, static):
    """Co-located stations should track each other. If they don't, suspect the datum."""
    rule("DUPLICATE STATION CHECK")
    rows = []
    for wb, grp in static.groupby("water_body"):
        if len(grp) < 2:
            continue
        ids = grp["gauge_id"].tolist()
        for i in range(len(ids)):
            for j in range(i + 1, len(ids)):
                a = daily[daily["gauge_id"] == ids[i]].set_index("date")[TARGET]
                b = daily[daily["gauge_id"] == ids[j]].set_index("date")[TARGET]
                joined = pd.concat([a, b], axis=1, join="inner").dropna()
                if len(joined) < 100:
                    continue
                r = joined.iloc[:, 0].corr(joined.iloc[:, 1])
                dr = joined.iloc[:, 0].diff().corr(joined.iloc[:, 1].diff())
                rows.append({"water_body": wb, "gauge_a": ids[i], "gauge_b": ids[j],
                             "n_days": len(joined), "corr_level": round(r, 4),
                             "corr_daily_change": round(dr, 4),
                             "mean_offset_m": round(
                                 (joined.iloc[:, 0] - joined.iloc[:, 1]).mean(), 3)})
    if not rows:
        print("    no co-located stations in this sample")
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    print("\n    corr_level near 1 = same signal. A large mean_offset_m with high")
    print("    correlation means different gauge datums, not different behaviour.")
    df.to_csv(OUT / "duplicate_check.csv", index=False)
    return df


# ------------------------------------------------------------ seasonal anomaly
def deseasonalise(s, dates):
    """
    Remove the day-of-year climatology.

    raw lake level and raw precipitation both carry a strong annual
    cycle, so their cross-correlation mostly measures 'both are seasonal' rather
    than any causal lag. Working on anomalies is what makes the CCF interpretable.
    """
    doy = dates.dt.dayofyear
    clim = s.groupby(doy).transform("mean")
    return s - clim


# ------------------------------------------------------------------------ CCF
def cross_correlation(target, driver, max_lag=MAX_LAG):
    """Correlate driver at t-k against target at t, for k = 0..max_lag."""
    out = []
    for k in range(max_lag + 1):
        d = driver.shift(k)
        pair = pd.concat([target, d], axis=1).dropna()
        if len(pair) < 200:
            out.append(np.nan)
            continue
        out.append(pair.iloc[:, 0].corr(pair.iloc[:, 1]))
    return np.array(out)


def run_ccf(daily, static):
    rule("CROSS-CORRELATION: DRIVERS vs LEVEL")
    ccf_rows, peak_rows = [], []

    for _, meta in static.iterrows():
        gid = int(meta["gauge_id"])
        g = daily[daily["gauge_id"] == gid].sort_values("date").copy()
        if len(g) < 1000:
            continue

        lvl_a = deseasonalise(g[TARGET], g["date"])
        for drv in DRIVERS:
            if drv not in g.columns or g[drv].notna().sum() < 1000:
                continue
            drv_a = deseasonalise(g[drv], g["date"])
            ccf = cross_correlation(lvl_a.reset_index(drop=True),
                                    drv_a.reset_index(drop=True))
            if np.all(np.isnan(ccf)):
                continue

            for k, v in enumerate(ccf):
                ccf_rows.append({"gauge_id": gid, "water_body": meta["water_body"],
                                 "driver": drv, "lag_days": k, "corr": v})

            peak_k = int(np.nanargmax(np.abs(ccf)))
            peak_rows.append({
                "gauge_id": gid,
                "water_body": meta["water_body"],
                "regulation_tier": meta.get("regulation_tier"),
                "driver": drv,
                "peak_lag_days": peak_k,
                "peak_corr": round(float(ccf[peak_k]), 4),
                "corr_at_lag0": round(float(ccf[0]), 4),
                "peak_sign": "positive" if ccf[peak_k] > 0 else "negative",
                # a peak within 5 days of the scan limit may be truncated:
                # the true maximum could lie beyond MAX_LAG
                "peak_at_boundary": peak_k >= MAX_LAG - 5,
            })

    ccf_df = pd.DataFrame(ccf_rows)
    peak_df = pd.DataFrame(peak_rows)
    ccf_df.to_csv(OUT / "ccf_results.csv", index=False)
    peak_df.to_csv(OUT / "peak_lags.csv", index=False)

    for drv in DRIVERS:
        sub = peak_df[peak_df["driver"] == drv]
        if len(sub):
            print(f"\n    {drv}")
            print(sub[["water_body", "peak_lag_days", "peak_corr", "peak_sign",
                       "peak_at_boundary"]]
                  .sort_values("peak_lag_days").to_string(index=False))
            edge = sub[sub["peak_at_boundary"]]
            if len(edge):
                print(f"    !! peak at scan limit for: {list(edge['water_body'])}")
                print(f"       True peak may exceed {MAX_LAG} days - say so, or raise MAX_LAG.")
    return ccf_df, peak_df


# -------------------------------------------------------------------- Granger
def run_granger(daily, static, maxlag=30):
    rule("GRANGER CAUSALITY (on stationary first differences)")
    try:
        from statsmodels.tsa.stattools import grangercausalitytests, adfuller
    except ImportError:
        print("    statsmodels not installed - skipping")
        return pd.DataFrame()

    rows = []
    for _, meta in static.iterrows():
        gid = int(meta["gauge_id"])
        g = daily[daily["gauge_id"] == gid].sort_values("date")
        for drv in DRIVERS:
            if drv not in g.columns:
                continue
            pair = pd.DataFrame({
                "y": g[TARGET].diff(),
                "x": g[drv].diff(),
            }).dropna()
            if len(pair) < 500 or pair["x"].std() == 0:
                continue
            try:
                adf_p = adfuller(pair["y"], autolag="AIC")[1]
                res = grangercausalitytests(pair[["y", "x"]], maxlag=maxlag)
                pvals = {k: v[0]["ssr_ftest"][1] for k, v in res.items()}
                best = min(pvals, key=pvals.get)
                rows.append({"gauge_id": gid, "water_body": meta["water_body"],
                             "driver": drv, "adf_p_target": round(adf_p, 5),
                             "best_lag": best, "min_p": pvals[best],
                             "significant_5pct": pvals[best] < 0.05})
            except Exception as e:
                print(f"    {gid} {drv}: {e}")

    df = pd.DataFrame(rows)
    if len(df):
        df.to_csv(OUT / "granger_results.csv", index=False)
        print(df.to_string(index=False))
        print("\n    Note: Granger causality is predictive precedence, not physical")
        print("    causation. Phrase it that way in the write-up.")
    return df


# --------------------------------------------------- does elevation drive lag?
def elevation_regression(peak_df, static):
    rule("PEAK LAG vs LAKE ATTRIBUTES")
    try:
        import statsmodels.api as sm
    except ImportError:
        print("    statsmodels not installed - skipping")
        return

    elev, elev_col = elevation_series(static)
    if elev_col:
        print(f"    elevation column resolved to: {elev_col}")
    else:
        print("    !! no elevation column found - the headline test cannot run")

    candidates = ELEV_NAMES + ["area", "catchment_area_km2", "frac_snow",
                               "glac_area", "p_mean", "aridity", "mean_slope"]
    have = [c for c in dict.fromkeys(candidates) if c in static.columns
            and static[c].notna().any()]
    print(f"    predictors available: {have}")

    lines = []
    for drv in DRIVERS:
        sub = peak_df[peak_df["driver"] == drv].merge(
            static[["gauge_id"] + have], on="gauge_id", how="left")
        sub = sub.dropna(subset=["peak_lag_days"])
        if len(sub) < 6:
            continue

        # Bivariate first - at n=14 this is the defensible result.
        header = f"\n### {drv}: peak lag vs each attribute   (n={len(sub)})"
        print(header)
        lines.append(header)
        for c in have:
            x = pd.to_numeric(sub[c], errors="coerce")
            if x.notna().sum() < 6 or x.std(skipna=True) == 0:
                continue
            r = sub["peak_lag_days"].astype(float).corr(x)
            rs = sub["peak_lag_days"].astype(float).corr(x, method="spearman")
            line = f"    r(peak_lag, {c:<20}) = {r:+.3f}   spearman = {rs:+.3f}"
            print(line)
            lines.append(line)

        # Sign of the peak correlation against elevation - the mechanism test.
        if elev_col and "peak_sign" in sub.columns:
            sub["_elev"] = sub["gauge_id"].map(elev)
            pos = sub[sub["peak_sign"] == "positive"]["_elev"].dropna()
            neg = sub[sub["peak_sign"] == "negative"]["_elev"].dropna()
            if len(pos) and len(neg):
                msg = (f"    peak correlation POSITIVE at mean elevation {pos.mean():.0f} m "
                       f"(n={len(pos)}), NEGATIVE at {neg.mean():.0f} m (n={len(neg)})")
                print(msg)
                lines.append(msg)

        preds = [c for c in have if pd.to_numeric(sub[c], errors="coerce").notna().sum()
                 >= len(sub) - 1 and pd.to_numeric(sub[c], errors="coerce").std() > 0]
        if elev_col and elev_col in preds:                 # keep elevation, drop others
            preds = [elev_col] + [p for p in preds if p != elev_col]
        preds = preds[:3]
        if not preds:
            continue

        Xv = sub[preds].apply(pd.to_numeric, errors="coerce")
        X = sm.add_constant(Xv.fillna(Xv.mean()))
        y = sub["peak_lag_days"].astype(float)
        model = sm.OLS(y, X).fit()
        txt = f"\n### OLS: peak lag of {drv} ~ {' + '.join(preds)}   (n={len(sub)})"
        print(txt)
        print(model.summary().as_text())
        lines.append(txt + "\n" + model.summary().as_text())

    (OUT / "elevation_regression.txt").write_text("\n\n".join(lines), encoding="utf-8")
    print(f"\n    -> {OUT / 'elevation_regression.txt'}")
    print("    With n around 14, treat multivariate coefficients as descriptive.")
    print("    The bivariate correlation is the defensible headline.")


# ------------------------------------------------------------------- figures
def make_figures(ccf_df, peak_df, static, daily):
    rule("FIGURES")
    FIG.mkdir(exist_ok=True)

    elev, elev_col = elevation_series(static)
    if len(elev.dropna()):
        lo, hi = float(np.nanmin(elev)), float(np.nanmax(elev))
    else:
        lo = hi = np.nan

    def shade_for(gid):
        e = elev.get(gid, np.nan)
        if not np.isfinite(e) or not np.isfinite(lo) or hi == lo:
            return 0.5, e
        return (e - lo) / (hi - lo), e

    # Fig 1: CCF curves, coloured by elevation
    for drv in DRIVERS:
        sub = ccf_df[ccf_df["driver"] == drv]
        if sub.empty:
            continue
        fig, ax = plt.subplots(figsize=(9, 5.5))
        for gid in sub["gauge_id"].unique():
            s = sub[sub["gauge_id"] == gid]
            shade, e = shade_for(gid)
            ax.plot(s["lag_days"], s["corr"], color=plt.cm.viridis(shade),
                    lw=1.4, alpha=0.85,
                    label=f"{s['water_body'].iloc[0]} ({e:.0f} m)"
                    if np.isfinite(e) else s["water_body"].iloc[0])
        ax.axhline(0, color="k", lw=0.6)
        ax.set_xlabel("Lag (days)")
        ax.set_ylabel("Correlation (seasonal anomalies)")
        ax.set_title(f"Cross-correlation: {drv} leading lake level")
        ax.legend(fontsize=6.5, ncol=2, frameon=False)
        fig.tight_layout()
        fig.savefig(FIG / f"ccf_{drv}.png", dpi=180)
        plt.close(fig)
        print(f"    figures/ccf_{drv}.png")

    # Fig 2: peak lag against elevation
    if len(elev):
        fig, axes = plt.subplots(1, len(DRIVERS), figsize=(4.2 * len(DRIVERS), 4), sharey=True)
        axes = np.atleast_1d(axes)
        for ax, drv in zip(axes, DRIVERS):
            sub = peak_df[peak_df["driver"] == drv]
            if sub.empty:
                continue
            x = [elev.get(g, np.nan) for g in sub["gauge_id"]]
            ax.scatter(x, sub["peak_lag_days"], s=45, alpha=0.8, edgecolor="k", lw=0.5)
            for xi, yi, nm in zip(x, sub["peak_lag_days"], sub["water_body"]):
                if np.isfinite(xi):
                    ax.annotate(nm[:11], (xi, yi), fontsize=5.5,
                                xytext=(3, 3), textcoords="offset points")
            ax.set_xlabel("Lake elevation (m)")
            ax.set_title(drv, fontsize=9)
        axes[0].set_ylabel("Peak-correlation lag (days)")
        fig.tight_layout()
        fig.savefig(FIG / "peak_lag_vs_elevation.png", dpi=180)
        plt.close(fig)
        print("    figures/peak_lag_vs_elevation.png")

    # Fig 3: annual cycle per lake, standardised so lakes are comparable
    fig, ax = plt.subplots(figsize=(9, 5))
    for gid, g in daily.groupby("gauge_id"):
        s = g.set_index("date")[TARGET]
        z = (s - s.mean()) / s.std()
        clim = z.groupby(z.index.dayofyear).mean()
        shade, _ = shade_for(gid)
        ax.plot(clim.index, clim.values, lw=1.3, alpha=0.85, color=plt.cm.viridis(shade))
    ax.set_xlabel("Day of year")
    ax.set_ylabel("Standardised level anomaly")
    ax.set_title("Annual cycle by lake (dark = low elevation, bright = high)")
    fig.tight_layout()
    fig.savefig(FIG / "annual_cycle.png", dpi=180)
    plt.close(fig)
    print("    figures/annual_cycle.png")


# -------------------------------------------------------------------- driver
def run_lag_analysis(daily, static, sample="N0", tag=None):
    OUT.mkdir(exist_ok=True)
    FIG.mkdir(exist_ok=True)
    tag = tag or (sample if isinstance(sample, str) else "custom")

    duplicate_check(daily, static)
    d, st = deduplicate(daily, static, sample=sample)
    ccf_df, peak_df = run_ccf(d, st)
    granger_df = run_granger(d, st)
    elevation_regression(peak_df, st)
    make_figures(ccf_df, peak_df, st, d)

    # keep each run's outputs separate
    st.to_csv(OUT / f"sample_{tag}.csv", index=False)
    for name in ("ccf_results", "peak_lags", "granger_results",
                 "elevation_regression", "duplicate_check"):
        for ext in (".csv", ".txt"):
            src = OUT / f"{name}{ext}"
            if src.exists():
                src.replace(OUT / f"{name}_{tag}{ext}")
    for p in FIG.glob("*.png"):
        if f"_{tag}" not in p.stem:
            p.replace(FIG / f"{p.stem}_{tag}.png")

    rule(f"DONE - RQ1 EVIDENCE COMPLETE (tag: {tag})")
    print(f"    outputs suffixed _{tag}")
    return {"daily": d, "static": st, "ccf": ccf_df,
            "peaks": peak_df, "granger": granger_df, "tag": tag}

In [11]:
"""
condense the RQ1 outputs into one short file you can paste.

Notebook:
    summarise(tags=["N0", "ALL"])

Writes: output/SUMMARY.txt   (deliberately compact - roughly 100 lines)

Reads whatever exists in ./output. Missing files are skipped, not fatal.
"""

from pathlib import Path

import numpy as np
import pandas as pd

OUT = Path("output")
ELEV_NAMES = ["mean_elevation_m", "gauge_elevation", "elev_mean", "elevation"]
DRIVERS = ["precipitation_mm", "temperature_mean_c", "snowmelt_proxy"]

L = []          # accumulated summary lines


def w(line=""):
    L.append(str(line))


def head(t):
    w("")
    w("=" * 70)
    w(t)
    w("=" * 70)


def load(name):
    p = OUT / name
    if not p.exists():
        return None
    try:
        return pd.read_csv(p)
    except Exception:
        return None


def elev_col(df):
    for c in ELEV_NAMES:
        if df is not None and c in df.columns and df[c].notna().any():
            return c
    if df is not None:
        for c in df.columns:
            if "elev" in c.lower() and df[c].notna().any():
                return c
    return None


# --------------------------------------------------------------- dataset facts
def summarise_dataset():
    head("DATASET")
    st = load("lakes_static.csv")
    qc = load("qc_report.csv")
    if st is None:
        w("lakes_static.csv missing")
        return
    ec = elev_col(st)
    w(f"stations           : {len(st)}")
    w(f"distinct water bodies: {st['water_body'].nunique()}")
    if "regulation_tier" in st:
        w(f"tiers              : {st['regulation_tier'].value_counts().to_dict()}")
    if ec:
        e = pd.to_numeric(st[ec], errors="coerce")
        w(f"elevation ({ec}) : {e.min():.0f} - {e.max():.0f} m, median {e.median():.0f}")
    if qc is not None:
        w(f"total spikes flagged : {int(qc['n_spike'].sum())}")
        w(f"total flatlines      : {int(qc['n_flatline'].sum())}")
        w(f"total long gaps      : {int(qc['n_gap_long'].sum())}")
        w(f"min completeness     : {qc['pct_usable_final'].min():.1f}%")

    for f in ("duplicate_check_N0.csv", "duplicate_check_ALL.csv", "duplicate_check.csv"):
        d = load(f)
        if d is not None and len(d):
            w("")
            w("co-located station agreement (level corr / daily-change corr):")
            for _, r in d.iterrows():
                w(f"  {r['water_body'][:22]:<22} {r['corr_level']:.4f} / "
                  f"{r['corr_daily_change']:.4f}  offset {r['mean_offset_m']:+.3f} m")
            break


# ------------------------------------------------------------------- per tag
def summarise_tag(tag):
    head(f"SAMPLE: {tag}")
    peaks = load(f"peak_lags_{tag}.csv") or load("peak_lags.csv")
    st = load(f"sample_{tag}.csv") or load("lakes_static.csv")
    gr = load(f"granger_results_{tag}.csv") or load("granger_results.csv")

    if peaks is None:
        w(f"no peak_lags for tag {tag}")
        return

    w(f"lakes analysed: {peaks['gauge_id'].nunique()}")
    ec = elev_col(st)
    elev = (pd.to_numeric(st.set_index("gauge_id")[ec], errors="coerce")
            if ec else pd.Series(dtype=float))

    for drv in DRIVERS:
        sub = peaks[peaks["driver"] == drv].copy()
        if sub.empty:
            continue
        sub["elev"] = sub["gauge_id"].map(elev)
        w("")
        w(f"--- {drv}")
        w(f"{'lake':<20}{'elev_m':>8}{'lag':>6}{'corr':>9}")
        for _, r in sub.sort_values("elev").iterrows():
            ev = f"{r['elev']:.0f}" if pd.notna(r.get("elev")) else "?"
            edge = " *" if r.get("peak_at_boundary") else ""
            w(f"{str(r['water_body'])[:19]:<20}{ev:>8}{int(r['peak_lag_days']):>6}"
              f"{r['peak_corr']:>9.3f}{edge}")

        # sign split - the mechanism test
        if sub["elev"].notna().any():
            pos = sub[sub["peak_corr"] > 0]["elev"].dropna()
            neg = sub[sub["peak_corr"] < 0]["elev"].dropna()
            if len(pos) and len(neg):
                w(f"  SIGN SPLIT: positive n={len(pos)} mean elev {pos.mean():.0f} m | "
                  f"negative n={len(neg)} mean elev {neg.mean():.0f} m")
            else:
                w(f"  all peaks {'positive' if len(pos) else 'negative'}")

        # bivariate correlations against every available attribute
        if st is not None:
            attrs = [c for c in (ELEV_NAMES + ["area", "catchment_area_km2", "frac_snow",
                                               "glac_area", "p_mean", "aridity", "mean_slope"])
                     if c in st.columns]
            m = sub.merge(st[["gauge_id"] + list(dict.fromkeys(attrs))],
                          on="gauge_id", how="left", suffixes=("", "_s"))
            bits = []
            for c in dict.fromkeys(attrs):
                x = pd.to_numeric(m[c], errors="coerce")
                if x.notna().sum() >= 6 and x.std() > 0:
                    r = m["peak_lag_days"].astype(float).corr(x)
                    bits.append(f"{c}={r:+.2f}")
            if bits:
                w("  r(lag, attr): " + "  ".join(bits))

    if gr is not None and len(gr):
        w("")
        w("--- Granger (first differences)")
        w(f"  tests run: {len(gr)}, significant at 5%: {int(gr['significant_5pct'].sum())}")
        for drv in DRIVERS:
            s = gr[gr["driver"] == drv]
            if len(s):
                w(f"  {drv:<20} median best lag = {s['best_lag'].median():.0f} d, "
                  f"range {s['best_lag'].min()}-{s['best_lag'].max()}")
        w("  (n~14600 per lake, so significance is near-automatic; report lag order)")


def summarise(tags=("N0", "ALL")):
    L.clear()
    w("RQ1 SUMMARY")
    summarise_dataset()
    for t in tags:
        summarise_tag(t)

    head("FILES PRESENT")
    for p in sorted(OUT.glob("*")):
        w(f"  {p.name:<38}{p.stat().st_size:>10,} bytes")

    text = "\n".join(L)
    (OUT / "SUMMARY.txt").write_text(text, encoding="utf-8")
    print(text)
    print(f"\n-> written: {OUT / 'SUMMARY.txt'}  ({len(L)} lines)")
    return text

In [ ]:
import contextlib, io
buf = io.StringIO()
with contextlib.redirect_stdout(buf):                 # keeps the notebook quiet
    res    = run_lag_analysis(daily, static, sample="N0",  tag="N0")
    res_all = run_lag_analysis(daily, static, sample="ALL", tag="ALL")
open("output/run_log.txt", "w", encoding="utf-8").write(buf.getvalue())

summarise(tags=["N0", "ALL"])

C:\Users\prave\anaconda3\Lib\site-packages\scipy\stats\_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=14 observations were given.
  return hypotest_fun_in(*args, **kwds)
C:\Users\prave\anaconda3\Lib\site-packages\scipy\stats\_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=14 observations were given.
  return hypotest_fun_in(*args, **kwds)
C:\Users\prave\anaconda3\Lib\site-packages\scipy\stats\_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=14 observations were given.
  return hypotest_fun_in(*args, **kwds)
C:\Users\prave\anaconda3\Lib\site-packages\scipy\stats\_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=14 observations were given.
  return hypotest_fun_in(*args, **kwds)
C:\Users\prave\anaconda3\Lib\sit

In [1]:
print(open("output/SUMMARY.txt", encoding="utf-8").read())

FileNotFoundError: [Errno 2] No such file or directory: 'output/SUMMARY.txt'

In [2]:
"""
03_summarise.py — condense the RQ1 outputs into one short file you can paste.

Notebook:
    summarise(tags=["N0", "ALL"])

Writes: output/SUMMARY.txt   (deliberately compact - roughly 100 lines)

Reads whatever exists in ./output. Missing files are skipped, not fatal.
"""

from pathlib import Path

import numpy as np
import pandas as pd

OUT = Path("output")
ELEV_NAMES = ["mean_elevation_m", "gauge_elevation", "elev_mean", "elevation"]
DRIVERS = ["precipitation_mm", "temperature_mean_c", "snowmelt_proxy"]

L = []          # accumulated summary lines


def w(line=""):
    L.append(str(line))


def head(t):
    w("")
    w("=" * 70)
    w(t)
    w("=" * 70)


def load(name):
    p = OUT / name
    if not p.exists():
        # tolerate tag variants: N0 vs NO, case differences
        alts = [q for q in OUT.glob("*")
                if q.name.lower().replace("o", "0") == name.lower().replace("o", "0")]
        if not alts:
            return None
        p = alts[0]
    try:
        return pd.read_csv(p)
    except Exception:
        return None


def first_of(*names):
    """First readable file among names. DataFrames are not truthy, so no 'or'."""
    for n in names:
        df = load(n)
        if df is not None and len(df):
            return df
    return None


def elev_col(df):
    for c in ELEV_NAMES:
        if df is not None and c in df.columns and df[c].notna().any():
            return c
    if df is not None:
        for c in df.columns:
            if "elev" in c.lower() and df[c].notna().any():
                return c
    return None


# --------------------------------------------------------------- dataset facts
def summarise_dataset():
    head("DATASET")
    st = load("lakes_static.csv")
    qc = load("qc_report.csv")
    if st is None:
        w("lakes_static.csv missing")
        return
    ec = elev_col(st)
    w(f"stations           : {len(st)}")
    w(f"distinct water bodies: {st['water_body'].nunique()}")
    if "regulation_tier" in st:
        w(f"tiers              : {st['regulation_tier'].value_counts().to_dict()}")
    if ec:
        e = pd.to_numeric(st[ec], errors="coerce")
        w(f"elevation ({ec}) : {e.min():.0f} - {e.max():.0f} m, median {e.median():.0f}")
    if qc is not None:
        w(f"total spikes flagged : {int(qc['n_spike'].sum())}")
        w(f"total flatlines      : {int(qc['n_flatline'].sum())}")
        w(f"total long gaps      : {int(qc['n_gap_long'].sum())}")
        w(f"min completeness     : {qc['pct_usable_final'].min():.1f}%")

    for f in ("duplicate_check_N0.csv", "duplicate_check_ALL.csv", "duplicate_check.csv"):
        d = load(f)
        if d is not None and len(d):
            w("")
            w("co-located station agreement (level corr / daily-change corr):")
            for _, r in d.iterrows():
                w(f"  {r['water_body'][:22]:<22} {r['corr_level']:.4f} / "
                  f"{r['corr_daily_change']:.4f}  offset {r['mean_offset_m']:+.3f} m")
            break


# ------------------------------------------------------------------- per tag
def summarise_tag(tag):
    head(f"SAMPLE: {tag}")
    peaks = first_of(f"peak_lags_{tag}.csv", "peak_lags.csv")
    st = first_of(f"sample_{tag}.csv", "lakes_static.csv")
    gr = first_of(f"granger_results_{tag}.csv", "granger_results.csv")

    if peaks is None:
        w(f"no peak_lags for tag {tag}")
        return

    w(f"lakes analysed: {peaks['gauge_id'].nunique()}")
    ec = elev_col(st)
    elev = (pd.to_numeric(st.set_index("gauge_id")[ec], errors="coerce")
            if ec else pd.Series(dtype=float))

    for drv in DRIVERS:
        sub = peaks[peaks["driver"] == drv].copy()
        if sub.empty:
            continue
        sub["elev"] = sub["gauge_id"].map(elev)
        w("")
        w(f"--- {drv}")
        w(f"{'lake':<20}{'elev_m':>8}{'lag':>6}{'corr':>9}")
        for _, r in sub.sort_values("elev").iterrows():
            ev = f"{r['elev']:.0f}" if pd.notna(r.get("elev")) else "?"
            edge = " *" if r.get("peak_at_boundary") else ""
            w(f"{str(r['water_body'])[:19]:<20}{ev:>8}{int(r['peak_lag_days']):>6}"
              f"{r['peak_corr']:>9.3f}{edge}")

        # sign split - the mechanism test
        if sub["elev"].notna().any():
            pos = sub[sub["peak_corr"] > 0]["elev"].dropna()
            neg = sub[sub["peak_corr"] < 0]["elev"].dropna()
            if len(pos) and len(neg):
                w(f"  SIGN SPLIT: positive n={len(pos)} mean elev {pos.mean():.0f} m | "
                  f"negative n={len(neg)} mean elev {neg.mean():.0f} m")
            else:
                w(f"  all peaks {'positive' if len(pos) else 'negative'}")

        # bivariate correlations against every available attribute
        if st is not None:
            attrs = [c for c in (ELEV_NAMES + ["area", "catchment_area_km2", "frac_snow",
                                               "glac_area", "p_mean", "aridity", "mean_slope"])
                     if c in st.columns]
            m = sub.merge(st[["gauge_id"] + list(dict.fromkeys(attrs))],
                          on="gauge_id", how="left", suffixes=("", "_s"))
            bits = []
            for c in dict.fromkeys(attrs):
                x = pd.to_numeric(m[c], errors="coerce")
                if x.notna().sum() >= 6 and x.std() > 0:
                    r = m["peak_lag_days"].astype(float).corr(x)
                    bits.append(f"{c}={r:+.2f}")
            if bits:
                w("  r(lag, attr): " + "  ".join(bits))

    if gr is not None and len(gr):
        w("")
        w("--- Granger (first differences)")
        w(f"  tests run: {len(gr)}, significant at 5%: {int(gr['significant_5pct'].sum())}")
        for drv in DRIVERS:
            s = gr[gr["driver"] == drv]
            if len(s):
                w(f"  {drv:<20} median best lag = {s['best_lag'].median():.0f} d, "
                  f"range {s['best_lag'].min()}-{s['best_lag'].max()}")
        w("  (n~14600 per lake, so significance is near-automatic; report lag order)")


def summarise(tags=("N0", "ALL")):
    L.clear()
    w("RQ1 SUMMARY")
    summarise_dataset()
    for t in tags:
        summarise_tag(t)

    head("FILES PRESENT")
    for p in sorted(OUT.glob("*")):
        w(f"  {p.name:<38}{p.stat().st_size:>10,} bytes")

    text = "\n".join(L)
    (OUT / "SUMMARY.txt").write_text(text, encoding="utf-8")
    print(text)
    print(f"\n-> written: {OUT / 'SUMMARY.txt'}  ({len(L)} lines)")
    return text

In [4]:
import warnings; warnings.filterwarnings("ignore")

exec(open("03_summarise.py", encoding="utf-8").read())   # or %run 03_summarise.py
summarise(tags=["N0", "ALL"])

RQ1 SUMMARY

DATASET
stations           : 31
distinct water bodies: 26
tiers              : {'N1': 16, 'N0': 15}
elevation (mean_elevation_m) : 191 - 1798 m, median 432
total spikes flagged : 4
total flatlines      : 0
total long gaps      : 2888
min completeness     : 94.0%

co-located station agreement (level corr / daily-change corr):
  Lac_Léman              0.9981 / 0.8801  offset +0.003 m
  Lac_de_Neuchâtel       0.9938 / 0.8397  offset +0.001 m
  Lago_Maggiore          0.9980 / 0.9553  offset +0.002 m
  Lago_di_Lugano         0.9835 / 0.8550  offset -0.009 m
  Vierwaldstättersee     0.9940 / 0.9344  offset -0.005 m

SAMPLE: N0
lakes analysed: 14

--- precipitation_mm
lake                  elev_m   lag     corr
Lago_di_Lugano           272     3    0.321
Zugersee                 413     9    0.191
Murtensee                431     4    0.297
Greifensee               440     4    0.301
Lauerzersee              448     3    0.375
Hallwilersee             448    12    0.178
Baldegger

"RQ1 SUMMARY\n\n======================================================================\nDATASET\n======================================================================\nstations           : 31\ndistinct water bodies: 26\ntiers              : {'N1': 16, 'N0': 15}\nelevation (mean_elevation_m) : 191 - 1798 m, median 432\ntotal spikes flagged : 4\ntotal flatlines      : 0\ntotal long gaps      : 2888\nmin completeness     : 94.0%\n\nco-located station agreement (level corr / daily-change corr):\n  Lac_Léman              0.9981 / 0.8801  offset +0.003 m\n  Lac_de_Neuchâtel       0.9938 / 0.8397  offset +0.001 m\n  Lago_Maggiore          0.9980 / 0.9553  offset +0.002 m\n  Lago_di_Lugano         0.9835 / 0.8550  offset -0.009 m\n  Vierwaldstättersee     0.9940 / 0.9344  offset -0.005 m\n\n======================================================================\nSAMPLE: N0\n======================================================================\nlakes analysed: 14\n\n--- precipitation_mm\nlake 

In [5]:
%run 04_models.py
res = run_models(sample_csv="output/sample_N0.csv", tag="N0")


PREPARING DATA
    static attributes: ['mean_elevation_m', 'mean_slope', 'frac_snow', 'glac_area', 'catchment_area_km2', 'p_mean', 'aridity']
    lakes prepared: 14

HORIZON h = 1 DAYS
    h=1: train 61,205  val 25,419  test 25,491
    fitting tabular models...
    feature matrix: (61205, 62)
    lightgbm not installed - skipped (pip install lightgbm)
    training LSTM...
    device: cuda
      epoch  1  val MSE 0.03763  <- best
      epoch  2  val MSE 0.02922  <- best
      epoch  3  val MSE 0.02797  <- best
      epoch  4  val MSE 0.02524  <- best
      epoch  5  val MSE 0.02555
      epoch  6  val MSE 0.02499  <- best
      epoch  7  val MSE 0.02260  <- best
      epoch  8  val MSE 0.02210  <- best
      epoch  9  val MSE 0.02217
      epoch 10  val MSE 0.02141  <- best
      epoch 11  val MSE 0.02076  <- best
      epoch 12  val MSE 0.02054  <- best
      epoch 13  val MSE 0.02193
      epoch 14  val MSE 0.02142
      epoch 15  val MSE 0.02172
      epoch 16  val MSE 0.02095
     

In [6]:
!pip install lightgbm

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 14.9 MB/s eta 0:00:00


In [8]:
import lightgbm as lgb
print("lightgbm", lgb.__version__)

import shutil, pathlib
for f in ["model_results_N0.csv", "model_strata_N0.csv"]:
    p = pathlib.Path("output") / f
    if p.exists():
        shutil.copy(p, p.with_suffix(".prev.csv"))
        print("backed up", f)

lightgbm 4.7.0
backed up model_results_N0.csv
backed up model_strata_N0.csv


In [9]:
%run 04_models.py
res = run_models(sample_csv="output/sample_N0.csv", tag="N0")


PREPARING DATA
    static attributes: ['mean_elevation_m', 'mean_slope', 'frac_snow', 'glac_area', 'catchment_area_km2', 'p_mean', 'aridity']
    lakes prepared: 14

HORIZON h = 1 DAYS
    h=1: train 61,205  val 25,419  test 25,491
    fitting tabular models...
    feature matrix: (61205, 62)


  File "C:\Users\prave\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\Users\prave\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\prave\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                        pass_fds, cwd, env,
                        ^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
                        gid, gids, uid, umask,
                        ^^^^^^^^^^^^^^^^^^^^^^
                        start_new_session, process_group)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\prave\anaconda3\Lib\subprocess.

    training LSTM...
    device: cuda
      epoch  1  val MSE 0.03763  <- best
      epoch  2  val MSE 0.02922  <- best
      epoch  3  val MSE 0.02797  <- best
      epoch  4  val MSE 0.02524  <- best
      epoch  5  val MSE 0.02555
      epoch  6  val MSE 0.02499  <- best
      epoch  7  val MSE 0.02260  <- best
      epoch  8  val MSE 0.02210  <- best
      epoch  9  val MSE 0.02217
      epoch 10  val MSE 0.02141  <- best
      epoch 11  val MSE 0.02076  <- best
      epoch 12  val MSE 0.02054  <- best
      epoch 13  val MSE 0.02193
      epoch 14  val MSE 0.02142
      epoch 15  val MSE 0.02172
      epoch 16  val MSE 0.02095
      epoch 17  val MSE 0.02178
      epoch 18  val MSE 0.02209
      early stop at epoch 18

HORIZON h = 3 DAYS
    h=3: train 61,167  val 25,417  test 25,489
    fitting tabular models...
    feature matrix: (61167, 62)
    training LSTM...
    device: cuda
      epoch  1  val MSE 0.11965  <- best
      epoch  2  val MSE 0.11082  <- best
      epoch  3  va

In [10]:
import pandas as pd

r = pd.read_csv("output/model_results_N0.csv")
s = pd.read_csv("output/model_strata_N0.csv")

print("=== RQ2: median across 14 lakes ===")
print(r.pivot_table(index="model", columns="horizon",
                    values=["pss", "nse", "rmse"], aggfunc="median").round(4).to_string())

print("\n=== RQ2: how often each model beats persistence (out of 14 lakes) ===")
win = r[r.model != "persistence"].copy()
win["beats"] = win["pss"] > 0
print(win.pivot_table(index="model", columns="horizon",
                      values="beats", aggfunc="sum").to_string())

print("\n=== RQ3: median PSS by season and extreme ===")
print(s.pivot_table(index=["model", "stratum"], columns="horizon",
                    values="pss", aggfunc="median").round(3).to_string())

with open("output/MODEL_SUMMARY.txt", "w", encoding="utf-8") as f:
    f.write(r.pivot_table(index="model", columns="horizon",
                          values=["pss","nse","rmse"], aggfunc="median").round(4).to_string())
    f.write("\n\n")
    f.write(s.pivot_table(index=["model","stratum"], columns="horizon",
                          values="pss", aggfunc="median").round(3).to_string())

=== RQ2: median across 14 lakes ===
                nse                     pss                    rmse                
horizon           1       3       7       1       3       7       1       3       7
model                                                                              
climatology  0.0675  0.0675  0.0675 -6.3564 -2.2109 -0.9783  0.1718  0.1718  0.1718
lightgbm     0.9896  0.9336  0.8016  0.3230  0.1792  0.0791  0.0193  0.0470  0.0866
lstm         0.9925  0.9339  0.8110  0.3542  0.1928  0.1221  0.0158  0.0464  0.0819
persistence  0.9818  0.9139  0.7566  0.0000  0.0000  0.0000  0.0243  0.0543  0.0937
ridge        0.9916  0.9188  0.7904  0.3490  0.1379  0.0717  0.0172  0.0496  0.0869

=== RQ2: how often each model beats persistence (out of 14 lakes) ===
horizon       1   3   7
model                  
climatology   0   0   0
lightgbm     13  13  11
lstm         12  13  13
ridge        13  11  12

=== RQ3: median PSS by season and extreme ===
horizon                       